# Experiment 28 — Exchange Expansion Across Three Strong Backbones

## 목적

Experiment 27에서 Solar에 적용한 **동일한 frozen historical-memory integration protocol**을
Exchange-rate benchmark로 확장합니다.

Backbones:

\[
\{\mathrm{PatchTST},\mathrm{iTransformer},\mathrm{TimeMixer}\}
\]

Forecast horizons:

\[
H\in\{96,192,336,720\}
\]

따라서 새로운 평가 조건은 \(3\times4=12\)개입니다.

---

# 중요한 해석상의 구분

Exchange는 기존 relevance 분석에서 Solar와 반대 성격을 보였던 데이터셋입니다.

- Solar: query-specific evidence가 강함
- Exchange: candidate-global / stable geometric prior가 강함

따라서 Experiment 28은 단순한 dataset 추가가 아니라,
**Solar에서 성공한 historical-memory integration이 다른 relevance regime에서도 strong forecaster를 개선하는지**를 확인하는 중요한 독립 확장 실험입니다.

Test 결과를 보기 전에 architecture와 integration rule은 변경하지 않습니다.

---

# Frozen historical-memory method

구조를 변경하지 않습니다.

\[
\boxed{\text{EmbeddingOnly Predictive Retrieval}
\rightarrow \text{Uniform Top-10 Historical Forecast}
\rightarrow \text{Cross-Fitted Adaptive Gate}
\rightarrow \text{Validation-Calibrated Shrinkage}}
\]

- retrieval lookback \(L_R=96\)
- Top-\(K=10\)
- Exchange memory stride 8
- 3 chronological OOF folds
- gate feature dimension 26
- shrinkage grid \(\{0,.25,.5,.75,1\}\)

Retriever architecture는 Solar 실험과 동일하게 유지합니다. Exchange checkpoint가 없으면, test 결과를 보기 전에 아래에 명시한 고정 listwise recipe로 full/fold retriever를 자동 학습한 뒤 고정합니다.

---

# Direct backbone policy

Experiment 27과 동일한 three-backbone recipe를 사용합니다.

### PatchTST
- direct lookback \(L_D=336\)
- patch length 16, stride 8
- \(d_{\mathrm{model}}=128\), 3 layers, 16 heads, \(d_{\mathrm{ff}}=256\)
- RevIN, Adam + OneCycleLR
- validation-only checkpoint selection

### iTransformer
- direct lookback \(L_D=96\)
- 3 layers, \(d_{\mathrm{model}}=512\), \(d_{\mathrm{ff}}=512\), 8 heads
- Adam, type-1 learning-rate schedule
- validation-only checkpoint selection

### TimeMixer
- direct lookback \(L_D=96\)
- channel independence
- 3 down-sampling layers, average pooling window 2
- \(d_{\mathrm{model}}=16\), \(d_{\mathrm{ff}}=32\), 3 layers
- Adam + OneCycleLR
- validation-only checkpoint selection

Exchange test 결과를 본 뒤 backbone별 hyperparameter를 바꾸지 않습니다.

---

# Data protocol

Standard Exchange-rate benchmark:

- 7,588 time points
- 8 channels
- daily sampling
- 70% train / 10% validation / 20% test
- train-only standardization
- all test origins, stride 1
- all channels
- \(H=\{96,192,336,720\}\)

Validation memory = train only. Test memory = train + validation only.
No rolling test-label memory is used.


In [1]:

from pathlib import Path
from types import SimpleNamespace
from contextlib import nullcontext

import gc
import importlib
import math
import os
import random
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings(
    "ignore"
)

pd.set_option(
    "display.max_columns",
    260,
)

pd.set_option(
    "display.width",
    560,
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DATASET = "Exchange"

BACKBONES = [
    "PatchTST",
    "iTransformer",
    "TimeMixer",
]

HORIZONS = [
    96,
    192,
    336,
    720,
]

RET_SEQ_LEN = 96

TOP_K = 10
MEMORY_STRIDE = 8
OOF_ANCHOR_STRIDE = 4

FOLDS = [
    (
        0.55,
        0.70,
    ),
    (
        0.70,
        0.85,
    ),
    (
        0.85,
        1.00,
    ),
]

# Frozen predictive representation.
REP_PATCH_LEN = 16
REP_PATCH_STRIDE = 16
REP_D_MODEL = 64
REP_N_HEADS = 4
REP_LAYERS = 2
REP_D_FF = 128
REP_DROPOUT = 0.1
REP_DIM = 64

REP_NUM_PATCHES = (
    1
    + (
        RET_SEQ_LEN
        - REP_PATCH_LEN
    )
    // REP_PATCH_STRIDE
)

# Prospectively frozen Exchange retriever training.
# These values are declared before any Exchange test metric is evaluated.
RETRIEVER_CANDIDATE_M = 100
RETRIEVER_TAU = 0.5
RETRIEVER_LR = 1e-3
RETRIEVER_WD = 1e-4
RETRIEVER_BATCH_QUERIES = 32
RETRIEVER_MAX_EPOCHS = 30
RETRIEVER_PATIENCE = 6
RETRIEVER_GRAD_CLIP = 5.0
RETRIEVER_QUERY_STRIDE = 4
RETRIEVER_MEMORY_FRACTION = 0.60
RETRIEVER_PHASEA_TRAIN_FRACTION = 0.80
RETRIEVER_SEED = 0

# Frozen gate.
GATE_DIM = 26
GATE_LR = 1e-3
GATE_WD = 1e-4
GATE_BATCH = 8192
GATE_MAX_EPOCHS = 50
GATE_PATIENCE = 7

ALPHA_GRID = np.round(
    np.arange(
        0.0,
        1.0001,
        0.1,
    ),
    10,
)

LAMBDA_GRID = np.array(
    [
        0.0,
        0.25,
        0.50,
        0.75,
        1.00,
    ],
    dtype=np.float32,
)

TARGET_RETRIEVAL_PAIRS = 672

EPS = 1e-8

RETRIEVER_USE_AMP = (
    torch.cuda.is_available()
)

CROSSFIT_SEED = 2828

RESUME = True
FORCE = False

BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_BLOCK_LEN = 24
BOOTSTRAP_SEED = 282800

ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "exchange_three_backbone_frozen_historical_memory"
)

ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SHARED_MEMORY_EMB_DIR = (
    ROOT
    / "shared_memory_embeddings"
)

SHARED_MEMORY_EMB_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ARTIFACT_DIR = (
    ROOT
    / "artifacts"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SUMMARY_PATH = (
    ROOT
    / "summary.csv"
)

BOOTSTRAP_PATH = (
    ROOT
    / "bootstrap.csv"
)

CURRENT_BACKBONE = None
DIRS = None

# Conservative direct inference blocks.
DIRECT_ANCHOR_BLOCK = {
    "PatchTST":
        16,
    "iTransformer":
        16,
    "TimeMixer":
        16,
}

print(
    "Device:",
    DEVICE,
)

print(
    "Output:",
    ROOT,
)


Device: cuda
Output: /data/dataset/strong_forecaster/exchange_three_backbone_frozen_historical_memory


## 1. Locate and load Exchange-rate

In [2]:
EXCHANGE_CANDIDATES = [
    Path("/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
    Path("/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv"),
    Path("/code/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
    Path("/data/dataset/exchange_rate/exchange_rate.csv"),
    Path("/data/dataset/exchange_rate.csv"),
]

EXCHANGE_PATH = next((p for p in EXCHANGE_CANDIDATES if p.is_file()), None)
if EXCHANGE_PATH is None:
    print("Attempted paths:")
    for p in EXCHANGE_CANDIDATES:
        print(" -", p)
    raise FileNotFoundError("Could not find Exchange-rate exchange_rate.csv.")


def load_exchange_csv(path):
    df = pd.read_csv(path)
    numeric = df.select_dtypes(include=[np.number]).copy()

    if numeric.shape[1] != 8:
        candidate = df.copy()
        for col in list(candidate.columns):
            if str(col).lower() in {"date", "datetime", "timestamp", "time"}:
                candidate = candidate.drop(columns=[col])
        candidate = candidate.apply(pd.to_numeric, errors="coerce")
        numeric = candidate.dropna(axis=1, how="all")

    numeric = (
        numeric.replace([np.inf, -np.inf], np.nan)
        .interpolate(axis=0, limit_direction="both")
        .ffill()
        .bfill()
    )

    if numeric.shape[1] != 8:
        raise ValueError(
            "Expected 8 Exchange channels, got "
            f"{numeric.shape[1]}. Columns={list(numeric.columns)}"
        )
    return numeric


raw_df = load_exchange_csv(EXCHANGE_PATH)
raw = raw_df.to_numpy(dtype=np.float32)
n_time, n_channels = raw.shape

if n_time != 7588:
    print(
        "WARNING: standard Exchange benchmark has 7,588 rows; "
        f"loaded {n_time}."
    )

train_end = int(0.70 * n_time)
num_test = int(0.20 * n_time)
val_end = n_time - num_test
test_end = n_time

train_mean = raw[:train_end].mean(axis=0).astype(np.float32)
train_std = raw[:train_end].std(axis=0, ddof=0).astype(np.float32)

if np.any(train_std <= 1e-6):
    bad = np.where(train_std <= 1e-6)[0]
    raise ValueError(f"Degenerate Exchange training channels: {bad[:20]}")

z_full = ((raw - train_mean[None, :]) / train_std[None, :]).astype(np.float32)

# Current manual forward wrappers do not consume time marks.
marks = np.zeros((n_time, 0), dtype=np.float32)

DATA = {
    "Exchange": {
        "name": "Exchange",
        "path": EXCHANGE_PATH,
        "raw": raw,
        "z": z_full,
        "marks": marks,
        "n_channels": n_channels,
        "train_end": train_end,
        "val_end": val_end,
        "test_end": test_end,
    }
}

display(pd.DataFrame([
    {"Split": "Train", "Start": 0, "EndExclusive": train_end, "Length": train_end},
    {"Split": "Validation", "Start": train_end, "EndExclusive": val_end, "Length": val_end-train_end},
    {"Split": "Test", "Start": val_end, "EndExclusive": test_end, "Length": test_end-val_end},
]))

print("Exchange path:", EXCHANGE_PATH)
print("Shape:", raw.shape)


,Split,Start,EndExclusive,Length
0,Train,0,5311,5311
1,Validation,5311,6071,760
2,Test,6071,7588,1517


Exchange path: /data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv
Shape: (7588, 8)


## 2. Prefix-only normalization for cross-fitting

In [3]:

def prefix_normalize(
    raw_array,
    prefix,
):
    prefix = int(
        prefix
    )

    mean = raw_array[
        :prefix
    ].mean(
        axis=0,
    ).astype(
        np.float32
    )

    std = raw_array[
        :prefix
    ].std(
        axis=0,
        ddof=0,
    ).astype(
        np.float32
    )

    if np.any(
        std
        <= 1e-6
    ):
        raise ValueError(
            f"Degenerate prefix channel at prefix={prefix}."
        )

    z = (
        (
            raw_array
            - mean[
                None,
                :
            ]
        )
        / std[
            None,
            :
        ]
    ).astype(
        np.float32
    )

    return (
        z,
        {
            "Mean":
                mean,
            "Std":
                std,
            "Prefix":
                prefix,
        },
    )


def eval_anchors(
    start,
    end,
    horizon,
    stride=1,
    lookback=RET_SEQ_LEN,
):
    return np.arange(
        max(
            int(
                start
            ),
            int(
                lookback
            ),
        ),
        int(
            end
        )
        - int(
            horizon
        )
        + 1,
        int(
            stride
        ),
        dtype=np.int64,
    )


## 3. Official backbone repositories

In [4]:

BASE = Path(
    "/code/stock_regime_retrieval/"
    "strong_forecaster"
)

REPO_CANDIDATES = {
    "PatchTST": [
        BASE
        / "PatchTST_official",
        Path(
            "/code/PatchTST_official"
        ),
        Path(
            "/data/PatchTST_official"
        ),
    ],

    "iTransformer": [
        BASE
        / "iTransformer_official",
        Path(
            "/code/iTransformer"
        ),
        Path(
            "/data/iTransformer"
        ),
    ],

    "TimeMixer": [
        BASE
        / "TimeMixer_official",
        Path(
            "/code/TimeMixer"
        ),
        Path(
            "/data/TimeMixer"
        ),
    ],
}

REPO_URLS = {
    "PatchTST":
        "https://github.com/yuqinie98/PatchTST.git",
    "iTransformer":
        "https://github.com/thuml/iTransformer.git",
    "TimeMixer":
        "https://github.com/kwuking/TimeMixer.git",
}

EXPECTED_FILES = {
    "PatchTST":
        Path(
            "PatchTST_supervised/"
            "models/PatchTST.py"
        ),
    "iTransformer":
        Path(
            "model/"
            "iTransformer.py"
        ),
    "TimeMixer":
        Path(
            "models/"
            "TimeMixer.py"
        ),
}

REPOS = {}

for backbone in BACKBONES:
    repo = next(
        (
            p
            for p in REPO_CANDIDATES[
                backbone
            ]
            if (
                p
                / EXPECTED_FILES[
                    backbone
                ]
            ).is_file()
        ),
        None,
    )

    if repo is None:
        repo = REPO_CANDIDATES[
            backbone
        ][
            0
        ]

        repo.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        print(
            f"Cloning {backbone} -> {repo}"
        )

        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                REPO_URLS[
                    backbone
                ],
                str(
                    repo
                ),
            ],
            check=True,
        )

    REPOS[
        backbone
    ] = repo

repo_rows = []

for backbone, repo in REPOS.items():
    try:
        commit = subprocess.check_output(
            [
                "git",
                "-C",
                str(
                    repo
                ),
                "rev-parse",
                "HEAD",
            ],
            text=True,
        ).strip()
    except Exception:
        commit = "unknown"

    repo_rows.append({
        "Backbone":
            backbone,
        "Repository":
            str(
                repo
            ),
        "Commit":
            commit,
    })

display(
    pd.DataFrame(
        repo_rows
    )
)

pd.DataFrame(
    repo_rows
).to_csv(
    ARTIFACT_DIR
    / "backbone_repository_commits.csv",
    index=False,
)


,Backbone,Repository,Commit
0,PatchTST,/code/stock_regime_retrieval/strong_forecaster...,204c21efe0b39603ad6e2ca640ef5896646ab1a9
1,iTransformer,/code/stock_regime_retrieval/strong_forecaster...,c2426e68ca13f74aaec08045c5c724d8ad328124
2,TimeMixer,/code/stock_regime_retrieval/strong_forecaster...,e24610583b36fdd8c76cc17a8df4e65759a5f460


## 4. Prespecified direct recipes

Exchange test metric을 보기 전에 아래 three-backbone recipe를 고정합니다.
Experiment 27의 Solar integration과 동일한 direct backbone recipe를 사용하여
**dataset만 변경하고 integration architecture는 변경하지 않는 독립 확장**으로 유지합니다.

이 Experiment 28 안에서는 test 결과를 근거로 recipe를 변경하지 않습니다.


In [5]:

DIRECT_RECIPES = {
    "PatchTST": {
        "seq_len":
            336,
        "label_len":
            48,
        "batch_size":
            32,
        "eval_batch":
            32,
        "train_epochs":
            100,
        "patience":
            10,
        "learning_rate":
            1e-4,
        "weight_decay":
            0.0,
        "scheduler":
            "TST",
        "pct_start":
            0.2,
        "seed":
            2021,

        "e_layers":
            3,
        "n_heads":
            16,
        "d_model":
            128,
        "d_ff":
            256,
        "dropout":
            0.2,
        "fc_dropout":
            0.2,
        "head_dropout":
            0.0,
        "patch_len":
            16,
        "stride":
            8,
    },

    "iTransformer": {
        "seq_len":
            96,
        "label_len":
            48,
        "batch_size":
            16,
        "eval_batch":
            16,
        "train_epochs":
            10,
        "patience":
            3,
        "learning_rate":
            5e-4,
        "weight_decay":
            0.0,
        "scheduler":
            "type1",
        "seed":
            2023,

        "e_layers":
            3,
        "n_heads":
            8,
        "d_model":
            512,
        "d_ff":
            512,
        "dropout":
            0.1,
        "factor":
            1,
    },

    "TimeMixer": {
        "seq_len":
            96,
        "label_len":
            0,
        "batch_size":
            128,
        "eval_batch":
            128,
        "train_epochs":
            20,
        "patience":
            10,
        "learning_rate":
            0.01,
        "weight_decay":
            0.0,
        "scheduler":
            "OneCycle",
        "pct_start":
            0.2,
        "seed":
            2021,

        "e_layers":
            3,
        "n_heads":
            8,
        "d_model":
            16,
        "d_ff":
            32,
        "dropout":
            0.1,
        "factor":
            3,

        "down_sampling_layers":
            3,
        "down_sampling_window":
            2,
        "down_sampling_method":
            "avg",
    },
}

display(
    pd.DataFrame(
        DIRECT_RECIPES
    ).T
)


,seq_len,label_len,batch_size,eval_batch,train_epochs,patience,learning_rate,weight_decay,scheduler,pct_start,seed,e_layers,n_heads,d_model,d_ff,dropout,fc_dropout,head_dropout,patch_len,stride,factor,down_sampling_layers,down_sampling_window,down_sampling_method
PatchTST,336,48,32,32,100,10,0.0001,0.0,TST,0.2,2021,3,16,128,256,0.2,0.2,0.0,16,8,NaN,NaN,NaN,NaN
iTransformer,96,48,16,16,10,3,0.0005,0.0,type1,NaN,2023,3,8,512,512,0.1,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN
TimeMixer,96,0,128,128,20,10,0.01,0.0,OneCycle,0.2,2021,3,8,16,32,0.1,NaN,NaN,NaN,NaN,3,3,2,avg


## 5. Backbone import activation

In [6]:

ACTIVE_MODEL_CLASS = None
ACTIVE_REPO = None


def clear_forecasting_modules():
    prefixes = [
        "models",
        "model",
        "layers",
        "utils",
        "data_provider",
        "exp",
        "experiments",
    ]

    for module_name in list(
        sys.modules.keys()
    ):
        if any(
            module_name
            == p
            or module_name.startswith(
                p
                + "."
            )
            for p in prefixes
        ):
            del sys.modules[
                module_name
            ]


def activate_backbone(
    backbone,
):
    global ACTIVE_MODEL_CLASS
    global ACTIVE_REPO

    clear_forecasting_modules()

    # Remove all known forecasting repositories from sys.path.
    repo_paths = []

    for b, r in REPOS.items():
        if b == "PatchTST":
            repo_paths.append(
                str(
                    r
                    / "PatchTST_supervised"
                )
            )
        else:
            repo_paths.append(
                str(
                    r
                )
            )

    sys.path[:] = [
        p
        for p in sys.path
        if p not in repo_paths
    ]

    if backbone == "PatchTST":
        root = (
            REPOS[
                backbone
            ]
            / "PatchTST_supervised"
        )

        sys.path.insert(
            0,
            str(
                root
            ),
        )

        module = importlib.import_module(
            "models.PatchTST"
        )

        ACTIVE_MODEL_CLASS = (
            module.Model
        )

        ACTIVE_REPO = root

    elif backbone == "iTransformer":
        root = REPOS[
            backbone
        ]

        sys.path.insert(
            0,
            str(
                root
            ),
        )

        module = importlib.import_module(
            "model.iTransformer"
        )

        ACTIVE_MODEL_CLASS = (
            module.Model
        )

        ACTIVE_REPO = root

    elif backbone == "TimeMixer":
        root = REPOS[
            backbone
        ]

        sys.path.insert(
            0,
            str(
                root
            ),
        )

        module = importlib.import_module(
            "models.TimeMixer"
        )

        ACTIVE_MODEL_CLASS = (
            module.Model
        )

        ACTIVE_REPO = root

    else:
        raise ValueError(
            backbone
        )

    print(
        f"Activated {backbone}: "
        f"{Path(module.__file__).resolve()}"
    )

    return ACTIVE_MODEL_CLASS


## 6. Direct-model construction and forward pass

In [7]:

def set_seed(
    seed,
):
    random.seed(
        int(
            seed
        )
    )

    np.random.seed(
        int(
            seed
        )
    )

    torch.manual_seed(
        int(
            seed
        )
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            int(
                seed
            )
        )

    torch.backends.cudnn.benchmark = False


def load_torch(
    path,
):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


def direct_config(
    backbone,
    horizon,
):
    r = DIRECT_RECIPES[
        backbone
    ]

    if backbone == "PatchTST":
        return SimpleNamespace(
            enc_in=
                n_channels,
            seq_len=
                r[
                    "seq_len"
                ],
            pred_len=
                int(
                    horizon
                ),
            e_layers=
                r[
                    "e_layers"
                ],
            n_heads=
                r[
                    "n_heads"
                ],
            d_model=
                r[
                    "d_model"
                ],
            d_ff=
                r[
                    "d_ff"
                ],
            dropout=
                r[
                    "dropout"
                ],
            fc_dropout=
                r[
                    "fc_dropout"
                ],
            head_dropout=
                r[
                    "head_dropout"
                ],
            individual=
                0,
            patch_len=
                r[
                    "patch_len"
                ],
            stride=
                r[
                    "stride"
                ],
            padding_patch=
                "end",
            revin=
                1,
            affine=
                0,
            subtract_last=
                0,
            decomposition=
                0,
            kernel_size=
                25,
        )

    if backbone == "iTransformer":
        return SimpleNamespace(
            task_name=
                "long_term_forecast",
            seq_len=
                r[
                    "seq_len"
                ],
            label_len=
                r[
                    "label_len"
                ],
            pred_len=
                int(
                    horizon
                ),
            enc_in=
                n_channels,
            dec_in=
                n_channels,
            c_out=
                n_channels,
            d_model=
                r[
                    "d_model"
                ],
            n_heads=
                r[
                    "n_heads"
                ],
            e_layers=
                r[
                    "e_layers"
                ],
            d_layers=
                1,
            d_ff=
                r[
                    "d_ff"
                ],
            moving_avg=
                25,
            factor=
                r[
                    "factor"
                ],
            distil=
                True,
            dropout=
                r[
                    "dropout"
                ],
            embed=
                "timeF",
            freq=
                "d",
            activation=
                "gelu",
            output_attention=
                False,
            use_norm=
                1,
            class_strategy=
                "projection",
        )

    if backbone == "TimeMixer":
        return SimpleNamespace(
            task_name=
                "long_term_forecast",
            seq_len=
                r[
                    "seq_len"
                ],
            label_len=
                r[
                    "label_len"
                ],
            pred_len=
                int(
                    horizon
                ),
            top_k=
                5,
            num_kernels=
                6,
            enc_in=
                n_channels,
            dec_in=
                n_channels,
            c_out=
                n_channels,
            d_model=
                r[
                    "d_model"
                ],
            n_heads=
                r[
                    "n_heads"
                ],
            e_layers=
                r[
                    "e_layers"
                ],
            d_layers=
                1,
            d_ff=
                r[
                    "d_ff"
                ],
            moving_avg=
                25,
            factor=
                r[
                    "factor"
                ],
            distil=
                True,
            dropout=
                r[
                    "dropout"
                ],
            embed=
                "timeF",
            freq=
                "d",
            activation=
                "gelu",
            output_attention=
                False,
            channel_independence=
                1,
            decomp_method=
                "moving_avg",
            use_norm=
                1,
            down_sampling_layers=
                r[
                    "down_sampling_layers"
                ],
            down_sampling_window=
                r[
                    "down_sampling_window"
                ],
            down_sampling_method=
                r[
                    "down_sampling_method"
                ],
            use_future_temporal_feature=
                0,
            features=
                "M",
        )

    raise ValueError(
        backbone
    )


def build_direct_model(
    backbone,
    horizon,
):
    if ACTIVE_MODEL_CLASS is None:
        raise RuntimeError(
            "Call activate_backbone() first."
        )

    cfg = direct_config(
        backbone,
        horizon,
    )

    model = ACTIVE_MODEL_CLASS(
        cfg
    ).float().to(
        DEVICE
    )

    return (
        model,
        cfg,
    )


def direct_seq_len(
    backbone,
):
    return int(
        DIRECT_RECIPES[
            backbone
        ][
            "seq_len"
        ]
    )


def make_direct_batch(
    z,
    anchors,
    backbone,
    horizon,
):
    anchors = np.asarray(
        anchors,
        dtype=np.int64,
    )

    L = direct_seq_len(
        backbone
    )

    x_idx = (
        anchors[
            :,
            None
        ]
        - L
        + np.arange(
            L
        )[
            None,
            :
        ]
    )

    y_idx = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    x = z[
        x_idx,
        :
    ].astype(
        np.float32
    )

    y = z[
        y_idx,
        :
    ].astype(
        np.float32
    )

    return (
        x,
        y,
    )


def forward_direct(
    backbone,
    model,
    x,
    y,
    horizon,
):
    x_t = torch.from_numpy(
        x
    ).to(
        DEVICE
    )

    y_t = torch.from_numpy(
        y
    ).to(
        DEVICE
    )

    if backbone == "PatchTST":
        pred = model(
            x_t
        )[
            :,
            -horizon:,
            :
        ]

    elif backbone == "iTransformer":
        label_len = DIRECT_RECIPES[
            backbone
        ][
            "label_len"
        ]

        dec_inp = torch.cat(
            [
                x_t[
                    :,
                    -label_len:,
                    :
                ],
                torch.zeros_like(
                    y_t
                ),
            ],
            dim=1,
        )

        pred = model(
            x_t,
            None,
            dec_inp,
            None,
        )[
            :,
            -horizon:,
            :
        ]

    elif backbone == "TimeMixer":
        pred = model(
            x_t,
            None,
            None,
            None,
        )[
            :,
            -horizon:,
            :
        ]

    else:
        raise ValueError(
            backbone
        )

    return (
        pred.float(),
        y_t.float(),
        x_t,
    )


@torch.no_grad()
def direct_residual_block(
    model,
    z,
    marks_unused,
    anchors,
    horizon,
):
    backbone = CURRENT_BACKBONE

    x, y = make_direct_batch(
        z,
        anchors,
        backbone,
        horizon,
    )

    pred, true, x_t = forward_direct(
        backbone,
        model,
        x,
        y,
        horizon,
    )

    current = x_t[
        :,
        -1:,
        :
    ].float()

    return (
        pred
        - current,
        true
        - current,
    )


## 7. Full direct baseline training

In [8]:

def backbone_dirs(
    backbone,
):
    root = (
        ROOT
        / backbone
    )

    dirs = {
        "root":
            root,
        "full_direct":
            root
            / "full_direct",
        "fold_direct":
            root
            / "fold_direct",
        "oof":
            root
            / "oof",
        "validation":
            root
            / "validation",
        "memory_emb":
            SHARED_MEMORY_EMB_DIR,
        "gate":
            root
            / "gate",
        "history":
            root
            / "history",
        "paired":
            root
            / "paired_test",
        "channel":
            root
            / "channel_test",
        "calibration":
            root
            / "calibration",
    }

    for p in dirs.values():
        if isinstance(
            p,
            Path,
        ):
            p.mkdir(
                parents=True,
                exist_ok=True,
            )

    return dirs


def full_direct_path(
    backbone,
    horizon,
):
    return (
        backbone_dirs(
            backbone
        )[
            "full_direct"
        ]
        / (
            f"Exchange_{backbone}_"
            f"H{horizon}.pt"
        )
    )


def full_direct_history_path(
    backbone,
    horizon,
):
    return (
        backbone_dirs(
            backbone
        )[
            "history"
        ]
        / (
            f"Exchange_{backbone}_"
            f"H{horizon}_full.csv"
        )
    )


def train_anchors_for_direct(
    backbone,
    boundary,
    horizon,
):
    L = direct_seq_len(
        backbone
    )

    return np.arange(
        L,
        int(
            boundary
        )
        - int(
            horizon
        )
        + 1,
        dtype=np.int64,
    )


@torch.no_grad()
def evaluate_direct_anchors(
    backbone,
    model,
    z,
    anchors,
    horizon,
    batch_size,
):
    model.eval()

    sse = 0.0
    sae = 0.0
    n = 0

    batch_mse = []

    for i in range(
        0,
        len(
            anchors
        ),
        batch_size,
    ):
        a = anchors[
            i:
            i+batch_size
        ]

        x, y = make_direct_batch(
            z,
            a,
            backbone,
            horizon,
        )

        pred, true, _ = forward_direct(
            backbone,
            model,
            x,
            y,
            horizon,
        )

        e = (
            pred
            - true
        )

        batch_mse.append(
            float(
                (
                    e
                    * e
                ).mean()
            )
        )

        sse += float(
            (
                e
                * e
            ).sum()
        )

        sae += float(
            e.abs().sum()
        )

        n += e.numel()

    return {
        "MSE":
            sse
            / n,
        "MAE":
            sae
            / n,
        "BatchAverageMSE":
            float(
                np.mean(
                    batch_mse
                )
            ),
        "Windows":
            len(
                anchors
            ),
    }


def set_epoch_lr_type1(
    optimizer,
    base_lr,
    epoch,
):
    lr = (
        base_lr
        * (
            0.5
            ** max(
                0,
                epoch
                - 1,
            )
        )
    )

    for group in optimizer.param_groups:
        group[
            "lr"
        ] = lr

    return lr


def train_or_load_full_direct(
    backbone,
    horizon,
):
    path = full_direct_path(
        backbone,
        horizon,
    )

    r = DIRECT_RECIPES[
        backbone
    ]

    model, cfg = build_direct_model(
        backbone,
        horizon,
    )

    if (
        RESUME
        and path.is_file()
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            f"Loaded full direct | "
            f"{backbone} H={horizon} | "
            f"best={ckpt['BestValMSE']:.6f}"
            f"@{ckpt['BestEpoch']}"
        )

        return (
            model,
            ckpt,
        )

    set_seed(
        r[
            "seed"
        ]
    )

    train_anchors = (
        train_anchors_for_direct(
            backbone,
            train_end,
            horizon,
        )
    )

    val_anchors = eval_anchors(
        train_end,
        val_end,
        horizon,
        stride=1,
        lookback=
            direct_seq_len(
                backbone
            ),
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=
            r[
                "learning_rate"
            ],
        weight_decay=
            r[
                "weight_decay"
            ],
    )

    scheduler = None

    if r[
        "scheduler"
    ] in {
        "TST",
        "OneCycle",
    }:
        steps_per_epoch = max(
            1,
            int(
                np.ceil(
                    len(
                        train_anchors
                    )
                    / r[
                        "batch_size"
                    ]
                )
            ),
        )

        scheduler = (
            torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                steps_per_epoch=
                    steps_per_epoch,
                pct_start=
                    r[
                        "pct_start"
                    ],
                epochs=
                    r[
                        "train_epochs"
                    ],
                max_lr=
                    r[
                        "learning_rate"
                    ],
            )
        )

    best_val = float(
        "inf"
    )

    best_epoch = -1
    best_state = None
    wait = 0
    history = []

    rng = np.random.default_rng(
        r[
            "seed"
        ]
        + horizon
    )

    for epoch in range(
        1,
        r[
            "train_epochs"
        ]
        + 1,
    ):
        model.train()

        if r[
            "scheduler"
        ] == "type1":
            current_lr = set_epoch_lr_type1(
                optimizer,
                r[
                    "learning_rate"
                ],
                epoch,
            )

        order = rng.permutation(
            train_anchors
        )

        losses = []
        t0 = time.time()

        for left in range(
            0,
            len(
                order
            ),
            r[
                "batch_size"
            ],
        ):
            a = order[
                left:
                left
                + r[
                    "batch_size"
                ]
            ]

            if len(
                a
            ) == 0:
                continue

            x, y = make_direct_batch(
                z_full,
                a,
                backbone,
                horizon,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            pred, true, _ = forward_direct(
                backbone,
                model,
                x,
                y,
                horizon,
            )

            loss = F.mse_loss(
                pred,
                true,
            )

            loss.backward()
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            losses.append(
                float(
                    loss.item()
                )
            )

        val = evaluate_direct_anchors(
            backbone,
            model,
            z_full,
            val_anchors,
            horizon,
            r[
                "eval_batch"
            ],
        )

        # Validation-only global MSE checkpoint selection.
        val_mse = val[
            "MSE"
        ]

        if (
            best_state is None
            or val_mse
            < best_val
            - 1e-12
        ):
            best_val = val_mse
            best_epoch = epoch

            best_state = {
                k:
                    v.detach()
                    .cpu()
                    .clone()
                for k, v
                in model.state_dict().items()
            }

            wait = 0
        else:
            wait += 1

        if scheduler is not None:
            current_lr = optimizer.param_groups[
                0
            ][
                "lr"
            ]

        history.append({
            "Epoch":
                epoch,
            "TrainMSE":
                float(
                    np.mean(
                        losses
                    )
                ),
            "ValMSE":
                val_mse,
            "ValMAE":
                val[
                    "MAE"
                ],
            "BestValMSE":
                best_val,
            "BestEpoch":
                best_epoch,
            "LR":
                current_lr,
            "Seconds":
                time.time()
                - t0,
        })

        pd.DataFrame(
            history
        ).to_csv(
            full_direct_history_path(
                backbone,
                horizon,
            ),
            index=False,
        )

        print(
            f"{backbone:12s} "
            f"H={horizon:3d} "
            f"ep={epoch:03d} | "
            f"train={history[-1]['TrainMSE']:.6f} | "
            f"val={val_mse:.6f} | "
            f"best={best_val:.6f}@{best_epoch} | "
            f"wait={wait}/{r['patience']}"
        )

        if (
            wait
            >= r[
                "patience"
            ]
        ):
            break

    if best_state is None:
        raise RuntimeError(
            f"No full direct checkpoint for "
            f"{backbone} H={horizon}."
        )

    model.load_state_dict(
        best_state
    )

    model.eval()

    ckpt = {
        "Backbone":
            backbone,
        "Dataset":
            "Exchange",
        "Horizon":
            int(
                horizon
            ),
        "SeqLen":
            direct_seq_len(
                backbone
            ),
        "BestEpoch":
            int(
                best_epoch
            ),
        "BestValMSE":
            float(
                best_val
            ),
        "Recipe":
            r,
        "StateDict":
            best_state,
    }

    torch.save(
        ckpt,
        path,
    )

    return (
        model,
        ckpt,
    )


## 8. Full direct baseline preflight

In [9]:

direct_baseline_rows = []

for backbone in BACKBONES:
    CURRENT_BACKBONE = (
        backbone
    )

    DIRS = backbone_dirs(
        backbone
    )

    activate_backbone(
        backbone
    )

    for horizon in HORIZONS:
        model, ckpt = (
            train_or_load_full_direct(
                backbone,
                horizon,
            )
        )

        test_anchors = eval_anchors(
            val_end,
            test_end,
            horizon,
            stride=1,
            lookback=
                direct_seq_len(
                    backbone
                ),
        )

        direct_test = (
            evaluate_direct_anchors(
                backbone,
                model,
                z_full,
                test_anchors,
                horizon,
                DIRECT_RECIPES[
                    backbone
                ][
                    "eval_batch"
                ],
            )
        )

        direct_baseline_rows.append({
            "Backbone":
                backbone,
            "Dataset":
                "Exchange",
            "Horizon":
                horizon,
            "SeqLen":
                direct_seq_len(
                    backbone
                ),
            "BestEpoch":
                ckpt[
                    "BestEpoch"
                ],
            "BestValMSE":
                ckpt[
                    "BestValMSE"
                ],
            "DirectTestMSE":
                direct_test[
                    "MSE"
                ],
            "DirectTestMAE":
                direct_test[
                    "MAE"
                ],
            "TestWindows":
                direct_test[
                    "Windows"
                ],
        })

        print(
            f"DIRECT | {backbone:12s} "
            f"H={horizon:3d} | "
            f"MSE={direct_test['MSE']:.6f} | "
            f"MAE={direct_test['MAE']:.6f}"
        )

        del (
            model,
            ckpt,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


direct_baseline_df = pd.DataFrame(
    direct_baseline_rows
)

direct_baseline_df.to_csv(
    ROOT
    / "direct_baselines.csv",
    index=False,
)

display(
    direct_baseline_df
)


Activated PatchTST: /code/stock_regime_retrieval/strong_forecaster/PatchTST_official/PatchTST_supervised/models/PatchTST.py
Loaded full direct | PatchTST H=96 | best=0.130612@6
DIRECT | PatchTST     H= 96 | MSE=0.088792 | MAE=0.208818
Loaded full direct | PatchTST H=192 | best=0.214889@8
DIRECT | PatchTST     H=192 | MSE=0.194937 | MAE=0.316183
Loaded full direct | PatchTST H=336 | best=0.364357@8
DIRECT | PatchTST     H=336 | MSE=0.346291 | MAE=0.428873
Loaded full direct | PatchTST H=720 | best=1.283822@9
DIRECT | PatchTST     H=720 | MSE=0.858390 | MAE=0.692987
Activated iTransformer: /code/stock_regime_retrieval/strong_forecaster/iTransformer_official/model/iTransformer.py
Loaded full direct | iTransformer H=96 | best=0.136243@2
DIRECT | iTransformer H= 96 | MSE=0.098471 | MAE=0.223430
Loaded full direct | iTransformer H=192 | best=0.213347@1
DIRECT | iTransformer H=192 | MSE=0.182288 | MAE=0.305967
Loaded full direct | iTransformer H=336 | best=0.361014@1
DIRECT | iTransformer H=3

,Backbone,Dataset,Horizon,SeqLen,BestEpoch,BestValMSE,DirectTestMSE,DirectTestMAE,TestWindows
0,PatchTST,Exchange,96,336,6,0.130612,0.088792,0.208818,1422
1,PatchTST,Exchange,192,336,8,0.214889,0.194937,0.316183,1326
2,PatchTST,Exchange,336,336,8,0.364357,0.346291,0.428873,1182
3,PatchTST,Exchange,720,336,9,1.283822,0.858390,0.692987,798
4,iTransformer,Exchange,96,96,2,0.136243,0.098471,0.223430,1422
5,iTransformer,Exchange,192,96,1,0.213347,0.182288,0.305967,1326
6,iTransformer,Exchange,336,96,1,0.361014,0.346023,0.426120,1182
7,iTransformer,Exchange,720,96,3,0.925093,0.832304,0.690283,798
8,TimeMixer,Exchange,96,96,4,0.130030,0.092543,0.210600,1422
9,TimeMixer,Exchange,192,96,3,0.226246,0.177509,0.299693,1326



## 9. Exchange predictive retriever checkpoints — auto-train if missing

Solar Experiment 27은 이미 존재하던 full/fold retriever checkpoint를 불러왔지만,
Exchange에는 해당 checkpoint가 아직 없을 수 있습니다. 이 버전은 이 경우 **중단하지 않고 자동으로 생성**합니다.

데이터 누수를 막기 위해 각 retriever는 자신이 사용될 시점 이전의 데이터만 사용합니다.

- full retriever: Exchange training split 내부만 사용
- fold retriever: 각 OOF prefix 이전 데이터만 사용
- validation/test future는 retriever 학습에 사용하지 않음
- 각 prefix의 earliest 60%를 retrieval memory로 두고, 이후 query-future pairs로 listwise supervision
- internal chronological validation으로 epoch 수만 선택한 뒤, 동일 prefix에서 처음부터 refit

고정 recipe:

- embedding-only predictive encoder: Solar와 동일한 architecture
- candidate pool: same-channel Pattern Top-100
- future target: query-wise standardized future MSE, temperature 0.5
- AdamW, lr=1e-3, weight decay=1e-4
- max 30 epochs, patience 6, gradient clipping 5.0
- query stride 4
- seed 0

**중요:** Exchange test metric을 계산하기 전에 이 recipe가 고정됩니다. 이후 결과를 보고 retriever hyperparameter를 변경하지 않습니다.


In [10]:

def inv_softplus(
    x,
):
    return math.log(
        math.exp(
            float(
                x
            )
        )
        - 1.0
    )


class PredictivePatchEncoder(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.patch_proj = nn.Linear(
            REP_PATCH_LEN,
            REP_D_MODEL,
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(
                1,
                REP_NUM_PATCHES,
                REP_D_MODEL,
            )
        )

        nn.init.trunc_normal_(
            self.pos_embed,
            std=0.02,
        )

        layer = nn.TransformerEncoderLayer(
            d_model=
                REP_D_MODEL,
            nhead=
                REP_N_HEADS,
            dim_feedforward=
                REP_D_FF,
            dropout=
                REP_DROPOUT,
            activation=
                "gelu",
            batch_first=
                True,
            norm_first=
                True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=
                REP_LAYERS,
        )

        self.norm = nn.LayerNorm(
            REP_D_MODEL
        )

        self.proj = nn.Linear(
            REP_D_MODEL,
            REP_DIM,
        )

    def forward(
        self,
        x,
    ):
        p = x.unfold(
            1,
            REP_PATCH_LEN,
            REP_PATCH_STRIDE,
        )

        h = (
            self.patch_proj(
                p
            )
            + self.pos_embed[
                :,
                :p.shape[
                    1
                ],
            ]
        )

        h = self.encoder(
            h
        ).mean(
            dim=1
        )

        h = self.proj(
            self.norm(
                h
            )
        )

        return F.normalize(
            h,
            dim=-1,
            eps=1e-8,
        )


class EmbeddingOnlyRetriever(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.encoder = (
            PredictivePatchEncoder()
        )

        self.raw_gamma = nn.Parameter(
            torch.tensor(
                inv_softplus(
                    1.0
                ),
                dtype=torch.float32,
            )
        )

    @property
    def gamma(
        self,
    ):
        return F.softplus(
            self.raw_gamma
        )

    def encode(
        self,
        x,
    ):
        return self.encoder(
            x
        )


def ret_amp():
    if RETRIEVER_USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()




STRONG_FORECASTER_ROOT = Path("/data/dataset/strong_forecaster")

EXP10_EXCHANGE_ROOT = STRONG_FORECASTER_ROOT / "exchange_predictive_representation"
EXP12_EXCHANGE_ROOT = STRONG_FORECASTER_ROOT / "exchange_crossfit_adaptive_gate"
EXP10_EXCHANGE_CKPT = EXP10_EXCHANGE_ROOT / "checkpoints"
EXP12_EXCHANGE_FOLD_RET = EXP12_EXCHANGE_ROOT / "fold_retriever_checkpoints"

_CKPT_RESOLVE_CACHE = {}


def _resolve_exchange_ckpt(filenames, preferred_dirs):
    key = tuple(str(x) for x in filenames)
    if key in _CKPT_RESOLVE_CACHE:
        return _CKPT_RESOLVE_CACHE[key]

    candidates = [Path(d) / fn for d in preferred_dirs for fn in filenames]
    for path in candidates:
        if path.is_file():
            _CKPT_RESOLVE_CACHE[key] = path
            return path

    matches = []
    if STRONG_FORECASTER_ROOT.is_dir():
        for fn in filenames:
            matches.extend(STRONG_FORECASTER_ROOT.rglob(fn))
    matches = sorted(set(p for p in matches if "exchange" in str(p).lower()))

    if len(matches) >= 1:
        if len(matches) > 1:
            print("WARNING: multiple Exchange checkpoint matches; using:", matches[0])
        _CKPT_RESOLVE_CACHE[key] = matches[0]
        return matches[0]

    path = candidates[0]
    _CKPT_RESOLVE_CACHE[key] = path
    return path


def full_retriever_ckpt_path(name, horizon):
    if name != "Exchange":
        raise ValueError(name)
    filenames = [
        f"Exchange_L96_H{horizon}_EmbeddingOnly_Listwise_seed0.pt",
        f"exchange_L96_H{horizon}_EmbeddingOnly_Listwise_seed0.pt",
    ]
    return _resolve_exchange_ckpt(
        filenames,
        [EXP10_EXCHANGE_CKPT, EXP10_EXCHANGE_ROOT],
    )


def fold_retriever_ckpt_path(name, horizon, fold):
    if name != "Exchange":
        raise ValueError(name)
    filenames = [
        f"Exchange_H{horizon}_Fold{fold}_EmbeddingOnly.pt",
        f"exchange_H{horizon}_Fold{fold}_EmbeddingOnly.pt",
    ]
    return _resolve_exchange_ckpt(
        filenames,
        [EXP12_EXCHANGE_FOLD_RET, EXP12_EXCHANGE_ROOT],
    )


def load_frozen_retriever(
    path,
):
    if not Path(
        path
    ).is_file():
        raise FileNotFoundError(
            path
        )

    ckpt = load_torch(
        path
    )

    model = EmbeddingOnlyRetriever().to(
        DEVICE
    )

    state = (
        ckpt[
            "StateDict"
        ]
        if isinstance(
            ckpt,
            dict,
        )
        and "StateDict"
        in ckpt
        else ckpt
    )

    # Backward-compatible checkpoint key handling.
    #
    # The frozen retriever checkpoints from the earlier experiments
    # were saved with:
    #   encoder.out_norm.* / encoder.out_proj.*
    # while this notebook's PredictivePatchEncoder currently exposes:
    #   encoder.norm.* / encoder.proj.*
    #
    # The modules are architecturally identical; only the attribute
    # names differ. Remap the legacy keys and still load strictly so
    # that any real architecture mismatch is not silently ignored.
    state = dict(
        state
    )

    legacy_to_current = {
        "encoder.out_norm.weight":
            "encoder.norm.weight",
        "encoder.out_norm.bias":
            "encoder.norm.bias",
        "encoder.out_proj.weight":
            "encoder.proj.weight",
        "encoder.out_proj.bias":
            "encoder.proj.bias",
    }

    for old_key, new_key in legacy_to_current.items():
        if (
            old_key in state
            and new_key not in state
        ):
            state[
                new_key
            ] = state.pop(
                old_key
            )

    model.load_state_dict(
        state,
        strict=True,
    )

    model.eval()

    for p in model.parameters():
        p.requires_grad_(
            False
        )

    return (
        model,
        ckpt,
    )


EXP10_EXCHANGE_CKPT.mkdir(parents=True, exist_ok=True)
EXP12_EXCHANGE_FOLD_RET.mkdir(parents=True, exist_ok=True)


In [11]:

# -----------------------------------------------------------------------------
# Auto-training for missing Exchange embedding-only retrievers.
# -----------------------------------------------------------------------------

def _ret_pattern_np(x):
    x = np.asarray(x, dtype=np.float32)
    xc = x - x.mean(axis=-1, keepdims=True)
    norm = np.linalg.norm(xc, axis=-1, keepdims=True)
    return np.where(norm > EPS, xc / np.maximum(norm, EPS), 0.0).astype(np.float32)


def _ret_extract_channel(z, channel, anchors, horizon):
    anchors = np.asarray(anchors, dtype=np.int64)
    pi = anchors[:, None] - RET_SEQ_LEN + np.arange(RET_SEQ_LEN)[None, :]
    fi = anchors[:, None] + np.arange(horizon)[None, :]
    past = z[pi, channel].astype(np.float32)
    future = z[fi, channel].astype(np.float32)
    current = z[anchors - 1, channel].astype(np.float32)
    future_residual = (future - current[:, None]).astype(np.float32)
    return past, future_residual


def _ret_soft_targets(dist):
    dist = np.asarray(dist, dtype=np.float32)
    mu = dist.mean(axis=1, keepdims=True)
    sd = dist.std(axis=1, keepdims=True)
    sd = np.maximum(sd, 1e-6)
    zdist = (dist - mu) / sd
    logits = -zdist / RETRIEVER_TAU
    logits = logits - logits.max(axis=1, keepdims=True)
    p = np.exp(logits).astype(np.float32)
    p /= np.maximum(p.sum(axis=1, keepdims=True), 1e-12)
    return p.astype(np.float32)


def _prepare_retriever_problem(z, prefix, horizon, query_stride=RETRIEVER_QUERY_STRIDE):
    """Build leakage-free same-channel Pattern Top-M listwise supervision."""
    prefix = int(prefix)
    horizon = int(horizon)
    C = z.shape[1]

    memory_end = int(RETRIEVER_MEMORY_FRACTION * prefix)
    memory_end = max(memory_end, RET_SEQ_LEN + horizon + MEMORY_STRIDE)

    memory_anchors = np.arange(
        RET_SEQ_LEN,
        memory_end - horizon + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )
    if len(memory_anchors) < TOP_K:
        raise ValueError(
            f"Too few memory candidates: prefix={prefix}, H={horizon}, M={len(memory_anchors)}"
        )

    m_use = min(RETRIEVER_CANDIDATE_M, len(memory_anchors))

    # Query origins are strictly after the fixed internal memory region.
    q_last = prefix - horizon
    if q_last < memory_end:
        raise ValueError(
            f"No retriever supervision interval: prefix={prefix}, H={horizon}, memory_end={memory_end}"
        )

    query_anchors = np.arange(
        memory_end,
        q_last + 1,
        query_stride,
        dtype=np.int64,
    )
    if len(query_anchors) < 4:
        raise ValueError(
            f"Too few retriever queries: prefix={prefix}, H={horizon}, n={len(query_anchors)}"
        )

    memory_past = np.empty((C, len(memory_anchors), RET_SEQ_LEN), dtype=np.float32)
    memory_pattern = np.empty_like(memory_past)
    memory_future = np.empty((C, len(memory_anchors), horizon), dtype=np.float32)

    for c in range(C):
        p, f = _ret_extract_channel(z, c, memory_anchors, horizon)
        memory_past[c] = p
        memory_pattern[c] = _ret_pattern_np(p)
        memory_future[c] = f

    # We flatten anchor x channel pairs.
    n_pairs = len(query_anchors) * C
    query_past = np.empty((n_pairs, RET_SEQ_LEN), dtype=np.float32)
    candidate_idx = np.empty((n_pairs, m_use), dtype=np.int32)
    candidate_channel = np.empty(n_pairs, dtype=np.int16)
    future_dist = np.empty((n_pairs, m_use), dtype=np.float32)
    pair_anchor = np.empty(n_pairs, dtype=np.int64)

    row = 0
    for c in range(C):
        qpast, qfuture = _ret_extract_channel(z, c, query_anchors, horizon)
        qpat = _ret_pattern_np(qpast)
        sim = qpat @ memory_pattern[c].T

        # Pattern Top-M: argpartition then exact descending order within the pool.
        if m_use < sim.shape[1]:
            idx0 = np.argpartition(sim, -m_use, axis=1)[:, -m_use:]
            score0 = np.take_along_axis(sim, idx0, axis=1)
            order = np.argsort(-score0, axis=1)
            idx = np.take_along_axis(idx0, order, axis=1)
        else:
            idx = np.argsort(-sim, axis=1)[:, :m_use]

        cand_future = memory_future[c][idx]
        dist = ((cand_future - qfuture[:, None, :]) ** 2).mean(axis=2).astype(np.float32)

        sl = slice(row, row + len(query_anchors))
        query_past[sl] = qpast
        candidate_idx[sl] = idx.astype(np.int32)
        candidate_channel[sl] = c
        future_dist[sl] = dist
        pair_anchor[sl] = query_anchors
        row += len(query_anchors)

    target_prob = _ret_soft_targets(future_dist)

    # Chronological Phase-A split by query origin, not by randomly shuffled rows.
    unique_a = np.unique(pair_anchor)
    cut_n = max(1, min(len(unique_a) - 1, int(RETRIEVER_PHASEA_TRAIN_FRACTION * len(unique_a))))
    cut_anchor = unique_a[cut_n]
    train_mask = pair_anchor < cut_anchor
    val_mask = pair_anchor >= cut_anchor

    if train_mask.sum() == 0 or val_mask.sum() == 0:
        raise ValueError(
            f"Invalid retriever internal split: prefix={prefix}, H={horizon}, "
            f"train={train_mask.sum()}, val={val_mask.sum()}"
        )

    return {
        "prefix": prefix,
        "memory_end": memory_end,
        "memory_anchors": memory_anchors,
        "memory_past": memory_past,
        "query_past": query_past,
        "candidate_idx": candidate_idx,
        "candidate_channel": candidate_channel,
        "future_dist": future_dist,
        "target_prob": target_prob,
        "pair_anchor": pair_anchor,
        "train_ids": np.where(train_mask)[0].astype(np.int64),
        "val_ids": np.where(val_mask)[0].astype(np.int64),
        "all_ids": np.arange(n_pairs, dtype=np.int64),
        "candidate_m": m_use,
    }


def _retriever_batch(model, problem, ids):
    ids = np.asarray(ids, dtype=np.int64)
    q_np = problem["query_past"][ids]
    ch = problem["candidate_channel"][ids].astype(np.int64)
    ci = problem["candidate_idx"][ids].astype(np.int64)
    cand_np = problem["memory_past"][ch[:, None], ci]
    tgt_np = problem["target_prob"][ids]

    q = torch.from_numpy(q_np).to(DEVICE)
    cand = torch.from_numpy(cand_np.reshape(-1, RET_SEQ_LEN)).to(DEVICE)
    target = torch.from_numpy(tgt_np).to(DEVICE)

    qemb = model.encode(q)
    cemb = model.encode(cand).reshape(len(ids), problem["candidate_m"], REP_DIM)
    score = model.gamma * torch.einsum("bd,bmd->bm", qemb, cemb)
    loss = -(target * F.log_softmax(score, dim=1)).sum(dim=1).mean()
    return loss, score


@torch.no_grad()
def _retriever_val_analog_mse(model, problem, ids):
    model.eval()
    total = 0.0
    count = 0
    for i in range(0, len(ids), RETRIEVER_BATCH_QUERIES):
        bid = ids[i:i + RETRIEVER_BATCH_QUERIES]
        q_np = problem["query_past"][bid]
        ch = problem["candidate_channel"][bid].astype(np.int64)
        ci = problem["candidate_idx"][bid].astype(np.int64)
        cand_np = problem["memory_past"][ch[:, None], ci]

        q = torch.from_numpy(q_np).to(DEVICE)
        cand = torch.from_numpy(cand_np.reshape(-1, RET_SEQ_LEN)).to(DEVICE)
        with ret_amp():
            qemb = model.encode(q)
            cemb = model.encode(cand).reshape(len(bid), problem["candidate_m"], REP_DIM)
        score = model.gamma.float() * torch.einsum("bd,bmd->bm", qemb.float(), cemb.float())
        k = min(TOP_K, score.shape[1])
        top = torch.topk(score, k, dim=1).indices.cpu().numpy()
        d = problem["future_dist"][bid]
        selected = np.take_along_axis(d, top, axis=1)
        total += float(selected.sum())
        count += selected.size
        del q, cand, qemb, cemb, score
    return total / max(count, 1)


def _fit_retriever_epochs(model, problem, ids, epochs, seed, verbose_prefix):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=RETRIEVER_LR,
        weight_decay=RETRIEVER_WD,
    )
    rng = np.random.default_rng(seed + 12345)
    hist = []
    for epoch in range(1, epochs + 1):
        model.train()
        order = np.asarray(ids, dtype=np.int64).copy()
        rng.shuffle(order)
        losses = []
        for i in range(0, len(order), RETRIEVER_BATCH_QUERIES):
            bid = order[i:i + RETRIEVER_BATCH_QUERIES]
            optimizer.zero_grad(set_to_none=True)
            loss, _ = _retriever_batch(model, problem, bid)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), RETRIEVER_GRAD_CLIP)
            optimizer.step()
            losses.append(float(loss.item()))
        mean_loss = float(np.mean(losses)) if losses else float("nan")
        hist.append(mean_loss)
        print(
            f"{verbose_prefix} ep={epoch:02d}/{epochs:02d} | "
            f"loss={mean_loss:.6f} | gamma={float(model.gamma.detach().cpu()):.4f}"
        )
    return hist


def train_or_load_exchange_retriever(path, z, prefix, horizon, tag):
    path = Path(path)
    if RESUME and path.is_file() and not FORCE:
        print("Loaded frozen retriever:", path)
        return path

    set_seed(RETRIEVER_SEED)
    problem = _prepare_retriever_problem(z, prefix, horizon)

    print(
        f"Retriever prep | {tag} | H={horizon} | prefix={prefix} | "
        f"memory={len(problem['memory_anchors'])}/ch | candidates={problem['candidate_m']} | "
        f"train_pairs={len(problem['train_ids'])} | val_pairs={len(problem['val_ids'])}"
    )

    # Phase A: chronological internal validation chooses epoch count only.
    model = EmbeddingOnlyRetriever().to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=RETRIEVER_LR,
        weight_decay=RETRIEVER_WD,
    )
    rng = np.random.default_rng(RETRIEVER_SEED + horizon + prefix)

    best_val = float("inf")
    best_epoch = 1
    wait = 0
    phase_a_history = []

    for epoch in range(1, RETRIEVER_MAX_EPOCHS + 1):
        model.train()
        order = problem["train_ids"].copy()
        rng.shuffle(order)
        losses = []

        for i in range(0, len(order), RETRIEVER_BATCH_QUERIES):
            bid = order[i:i + RETRIEVER_BATCH_QUERIES]
            optimizer.zero_grad(set_to_none=True)
            loss, _ = _retriever_batch(model, problem, bid)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), RETRIEVER_GRAD_CLIP)
            optimizer.step()
            losses.append(float(loss.item()))

        val_mse = _retriever_val_analog_mse(model, problem, problem["val_ids"])
        train_loss = float(np.mean(losses)) if losses else float("nan")
        phase_a_history.append((epoch, train_loss, val_mse, float(model.gamma.detach().cpu())))

        print(
            f"Retriever {tag:>8s} H={horizon:>3d} ep={epoch:02d} | "
            f"loss={train_loss:.6f} | valAnalog={val_mse:.6f} | "
            f"gamma={float(model.gamma.detach().cpu()):.4f}"
        )

        if val_mse < best_val - 1e-10:
            best_val = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1
            if wait >= RETRIEVER_PATIENCE:
                break

    del model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Phase B: reinitialize and refit on every historical query available before prefix.
    set_seed(RETRIEVER_SEED)
    model = EmbeddingOnlyRetriever().to(DEVICE)
    phase_b_loss = _fit_retriever_epochs(
        model,
        problem,
        problem["all_ids"],
        best_epoch,
        RETRIEVER_SEED + horizon + prefix,
        f"Refit {tag} H={horizon}",
    )

    path.parent.mkdir(parents=True, exist_ok=True)
    ckpt = {
        "StateDict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "Dataset": "Exchange",
        "Horizon": int(horizon),
        "Prefix": int(prefix),
        "Tag": str(tag),
        "BestEpoch": int(best_epoch),
        "BestValAnalogMSE": float(best_val),
        "Gamma": float(model.gamma.detach().cpu()),
        "Recipe": {
            "retrieval_seq_len": RET_SEQ_LEN,
            "memory_stride": MEMORY_STRIDE,
            "candidate_m": int(problem["candidate_m"]),
            "top_k": TOP_K,
            "target_temperature": RETRIEVER_TAU,
            "optimizer": "AdamW",
            "learning_rate": RETRIEVER_LR,
            "weight_decay": RETRIEVER_WD,
            "batch_queries": RETRIEVER_BATCH_QUERIES,
            "max_epochs": RETRIEVER_MAX_EPOCHS,
            "patience": RETRIEVER_PATIENCE,
            "gradient_clip": RETRIEVER_GRAD_CLIP,
            "query_stride": RETRIEVER_QUERY_STRIDE,
            "internal_memory_fraction": RETRIEVER_MEMORY_FRACTION,
            "phase_a_train_fraction": RETRIEVER_PHASEA_TRAIN_FRACTION,
            "seed": RETRIEVER_SEED,
            "candidate_rule": "same-channel Pattern Top-M",
            "target_rule": "query-wise standardized future MSE softmax",
        },
        "PhaseAHistory": phase_a_history,
        "PhaseBLoss": phase_b_loss,
    }
    torch.save(ckpt, path)
    print(
        f"Saved frozen retriever | {tag} H={horizon} | "
        f"bestEpoch={best_epoch} | valAnalog={best_val:.6f} | {path}"
    )

    del model, problem
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return path


def ensure_exchange_retriever_checkpoints():
    # Full retrievers use train-only normalization and train-only supervision.
    for horizon in HORIZONS:
        train_or_load_exchange_retriever(
            full_retriever_ckpt_path("Exchange", horizon),
            DATA["Exchange"]["z"],
            DATA["Exchange"]["train_end"],
            horizon,
            "Full",
        )

        # Fold retrievers are prefix-only; each is trained before its OOF interval begins.
        for fold, (p0, _p1) in enumerate(FOLDS, start=1):
            prefix = int(p0 * DATA["Exchange"]["train_end"])
            z_prefix, _ = prefix_normalize(DATA["Exchange"]["raw"], prefix)
            train_or_load_exchange_retriever(
                fold_retriever_ckpt_path("Exchange", horizon, fold),
                z_prefix,
                prefix,
                horizon,
                f"Fold{fold}",
            )
            del z_prefix
            gc.collect()


ensure_exchange_retriever_checkpoints()

# Final preflight after auto-training.
retriever_preflight_rows = []
for horizon in HORIZONS:
    full_path = full_retriever_ckpt_path("Exchange", horizon)
    retriever_preflight_rows.append({
        "Horizon": horizon,
        "Kind": "Full",
        "Path": str(full_path),
        "Exists": full_path.is_file(),
    })
    for fold in range(1, 4):
        fold_path = fold_retriever_ckpt_path("Exchange", horizon, fold)
        retriever_preflight_rows.append({
            "Horizon": horizon,
            "Kind": f"Fold{fold}",
            "Path": str(fold_path),
            "Exists": fold_path.is_file(),
        })

retriever_preflight = pd.DataFrame(retriever_preflight_rows)
display(retriever_preflight)
missing = retriever_preflight[~retriever_preflight["Exists"]]
if len(missing):
    display(missing)
    raise FileNotFoundError("Exchange retriever auto-training did not produce all required checkpoints.")
print("PASS: all 4 full + 12 fold Exchange retriever checkpoints exist.")


Retriever prep | Full | H=96 | prefix=5311 | memory=375/ch | candidates=100 | train_pairs=3248 | val_pairs=816
Retriever     Full H= 96 ep=01 | loss=4.605019 | valAnalog=0.284273 | gamma=0.9947
Retriever     Full H= 96 ep=02 | loss=4.602314 | valAnalog=0.278454 | gamma=0.9935
Retriever     Full H= 96 ep=03 | loss=4.601802 | valAnalog=0.289105 | gamma=0.9919
Retriever     Full H= 96 ep=04 | loss=4.599173 | valAnalog=0.266284 | gamma=0.9920
Retriever     Full H= 96 ep=05 | loss=4.598203 | valAnalog=0.243077 | gamma=0.9955
Retriever     Full H= 96 ep=06 | loss=4.597859 | valAnalog=0.284947 | gamma=1.0024
Retriever     Full H= 96 ep=07 | loss=4.596969 | valAnalog=0.233240 | gamma=1.0029
Retriever     Full H= 96 ep=08 | loss=4.594711 | valAnalog=0.228658 | gamma=1.0137
Retriever     Full H= 96 ep=09 | loss=4.593925 | valAnalog=0.230987 | gamma=1.0237
Retriever     Full H= 96 ep=10 | loss=4.592685 | valAnalog=0.245284 | gamma=1.0364
Retriever     Full H= 96 ep=11 | loss=4.591622 | valAnalog=

,Horizon,Kind,Path,Exists
0,96,Full,/data/dataset/strong_forecaster/exchange_predi...,True
1,96,Fold1,/data/dataset/strong_forecaster/exchange_cross...,True
2,96,Fold2,/data/dataset/strong_forecaster/exchange_cross...,True
3,96,Fold3,/data/dataset/strong_forecaster/exchange_cross...,True
4,192,Full,/data/dataset/strong_forecaster/exchange_predi...,True
5,192,Fold1,/data/dataset/strong_forecaster/exchange_cross...,True
6,192,Fold2,/data/dataset/strong_forecaster/exchange_cross...,True
7,192,Fold3,/data/dataset/strong_forecaster/exchange_cross...,True
8,336,Full,/data/dataset/strong_forecaster/exchange_predi...,True
9,336,Fold1,/data/dataset/strong_forecaster/exchange_cross...,True


PASS: all 4 full + 12 fold Exchange retriever checkpoints exist.


## 10. Retrieval time-series utilities

In [12]:

def retrieval_anchor_batch(
    name,
):
    C = DATA[
        name
    ][
        "n_channels"
    ]

    return max(
        1,
        TARGET_RETRIEVAL_PAIRS
        // C,
    )


def batch_pattern(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    xc = (
        x
        - x.mean(
            axis=-1,
            keepdims=True,
        )
    )

    n = np.linalg.norm(
        xc,
        axis=-1,
        keepdims=True,
    )

    return np.where(
        n > EPS,
        xc
        / np.maximum(
            n,
            EPS,
        ),
        0.0,
    ).astype(
        np.float32
    )


def context7(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    short = max(
        8,
        RET_SEQ_LEN
        // 4,
    )

    m = x.mean(
        axis=-1
    )

    s = (
        x.std(
            axis=-1
        )
        + EPS
    )

    f1 = (
        x[
            ...,
            -1
        ]
        - m
    ) / s

    f2 = (
        x[
            ...,
            -short:
        ].mean(
            axis=-1
        )
        - m
    ) / s

    f3 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            -short
        ]
    ) / s

    f4 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            0
        ]
    ) / s

    df = np.diff(
        x,
        axis=-1,
    )

    ds = np.diff(
        x[
            ...,
            -short:
        ],
        axis=-1,
    )

    f5 = (
        ds.std(
            axis=-1
        )
        + EPS
    ) / (
        df.std(
            axis=-1
        )
        + EPS
    )

    t = np.linspace(
        -1.0,
        1.0,
        RET_SEQ_LEN,
        dtype=np.float32,
    )

    t = (
        t
        - t.mean()
    )

    f6 = (
        np.sum(
            t
            * (
                x
                - m[
                    ...,
                    None
                ]
            ),
            axis=-1,
        )
        / (
            np.sum(
                t
                * t
            )
            + EPS
        )
    ) / s

    a = x[
        ...,
        :-1
    ]

    b = x[
        ...,
        1:
    ]

    a = (
        a
        - a.mean(
            axis=-1,
            keepdims=True,
        )
    )

    b = (
        b
        - b.mean(
            axis=-1,
            keepdims=True,
        )
    )

    f7 = np.sum(
        a
        * b,
        axis=-1,
    ) / (
        np.sqrt(
            np.sum(
                a
                * a,
                axis=-1,
            )
            * np.sum(
                b
                * b,
                axis=-1,
            )
        )
        + EPS
    )

    return np.stack(
        [
            f1,
            f2,
            f3,
            f4,
            f5,
            f6,
            f7,
        ],
        axis=-1,
    ).astype(
        np.float32
    )


def extract_channel(
    z,
    c,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        c,
    ].astype(
        np.float32
    )

    future = z[
        fi,
        c,
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        c,
    ].astype(
        np.float32
    )

    future_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        future_residual,
    )


def build_memory(
    z,
    channels,
    boundary,
    horizon,
):
    memory_anchors = np.arange(
        RET_SEQ_LEN,
        int(
            boundary
        )
        - horizon
        + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )

    if len(
        memory_anchors
    ) < TOP_K:
        raise ValueError(
            "Insufficient admissible memory."
        )

    past = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            RET_SEQ_LEN,
        ),
        dtype=np.float32,
    )

    pattern = np.empty_like(
        past
    )

    future = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            horizon,
        ),
        dtype=np.float32,
    )

    for c in range(
        channels
    ):
        p, f = extract_channel(
            z,
            c,
            memory_anchors,
            horizon,
        )

        past[
            c
        ] = p

        pattern[
            c
        ] = batch_pattern(
            p
        )

        future[
            c
        ] = f

    return {
        "anchors":
            memory_anchors,
        "past":
            past,
        "pattern":
            pattern,
        "future":
            future,
        "M":
            len(
                memory_anchors
            ),
        "boundary":
            int(
                boundary
            ),
    }


def query_pairs(
    z,
    anchors,
    channels,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    channels = np.asarray(
        channels,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    future = z[
        fi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        channels,
    ].astype(
        np.float32
    )

    true_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        batch_pattern(
            past
        ),
        context7(
            past
        ),
        true_residual,
    )


## 11. Shared cached memory embeddings

In [13]:

@torch.no_grad()
def encode_np(
    model,
    x,
    chunk=512,
):
    parts = []

    for i in range(
        0,
        len(
            x
        ),
        chunk,
    ):
        t = torch.from_numpy(
            x[
                i:
                i+chunk
            ]
        ).to(
            DEVICE
        )

        with ret_amp():
            e = model.encode(
                t
            ).float()

        parts.append(
            e.cpu()
        )

        del (
            t,
            e,
        )

    return torch.cat(
        parts,
        dim=0,
    ).numpy().astype(
        np.float32
    )


def memory_embedding_path(
    name,
    horizon,
    tag,
):
    return (
        SHARED_MEMORY_EMB_DIR
        / (
            f"{name}_H{horizon}_"
            f"{tag}_emb.npy"
        )
    )


@torch.no_grad()
def memory_gpu_cached(
    name,
    horizon,
    tag,
    model,
    memory,
    channels,
):
    path = memory_embedding_path(
        name,
        horizon,
        tag,
    )

    expected = (
        channels,
        memory[
            "M"
        ],
        REP_DIM,
    )

    emb_np = None

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        candidate = np.load(
            path,
            mmap_mode=None,
        )

        if (
            tuple(
                candidate.shape
            )
            == expected
        ):
            emb_np = candidate.astype(
                np.float32,
                copy=False,
            )

            print(
                "Loaded memory embedding:",
                path.name,
            )

    if emb_np is None:
        emb_np = np.empty(
            expected,
            dtype=np.float32,
        )

        print(
            "Building memory embedding:",
            path.name,
            expected,
        )

        for c in range(
            channels
        ):
            emb_np[
                c
            ] = encode_np(
                model,
                memory[
                    "past"
                ][
                    c
                ],
            )

            if (
                c == 0
                or (
                    c + 1
                )
                % 50
                == 0
                or (
                    c + 1
                    == channels
                )
            ):
                print(
                    f"  channel "
                    f"{c+1}/{channels}"
                )

        np.save(
            path,
            emb_np,
        )

    return {
        "emb":
            torch.from_numpy(
                emb_np
            ).to(
                DEVICE
            ),
        "pattern":
            torch.from_numpy(
                memory[
                    "pattern"
                ]
            ).to(
                DEVICE
            ),
        "future":
            torch.from_numpy(
                memory[
                    "future"
                ]
            ).to(
                DEVICE
            ),
    }


## 12. Frozen retrieval inference

In [14]:

@torch.no_grad()
def retrieve(
    model,
    memory_gpu_obj,
    z,
    anchors,
    channels,
    horizon,
):
    (
        past,
        pattern,
        ctx,
        true,
    ) = query_pairs(
        z,
        anchors,
        channels,
        horizon,
    )

    past_t = torch.from_numpy(
        past
    ).to(
        DEVICE
    )

    pattern_t = torch.from_numpy(
        pattern
    ).to(
        DEVICE
    )

    with ret_amp():
        qemb = model.encode(
            past_t
        )

    qemb = qemb.float()

    memb = memory_gpu_obj[
        "emb"
    ][
        channels
    ]

    sim = torch.bmm(
        qemb[
            :,
            None,
            :
        ],
        memb.transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    score = (
        model.gamma
        * sim
    )

    idx = torch.topk(
        score,
        TOP_K,
        dim=1,
    ).indices

    row = torch.arange(
        len(
            channels
        ),
        device=DEVICE,
    )[
        :,
        None
    ]

    pfull = torch.bmm(
        pattern_t[
            :,
            None,
            :
        ],
        memory_gpu_obj[
            "pattern"
        ][
            channels
        ].transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    return {
        "score":
            score[
                row,
                idx
            ],
        "sim":
            sim[
                row,
                idx
            ],
        "pattern":
            pfull[
                row,
                idx
            ],
        "cand":
            memory_gpu_obj[
                "future"
            ][
                channels[
                    :,
                    None
                ],
                idx,
            ],
        "ctx":
            torch.from_numpy(
                ctx
            ).to(
                DEVICE
            ),
        "true":
            torch.from_numpy(
                true
            ).to(
                DEVICE
            ),
    }


## 13. Frozen 26-dimensional adaptive gate

In [15]:

class CrossFitAdaptiveGate(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                GATE_DIM,
                64,
            ),
            nn.LayerNorm(
                64
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                64,
                32,
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                32,
                1,
            ),
        )

        nn.init.normal_(
            self.net[
                -1
            ].weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.constant_(
            self.net[
                -1
            ].bias,
            math.log(
                0.1
                / 0.9
            ),
        )

    def forward(
        self,
        x,
    ):
        return torch.sigmoid(
            self.net(
                x
            ).squeeze(
                -1
            )
        )


def score_entropy(
    s,
):
    p = torch.softmax(
        s,
        dim=1,
    )

    return (
        -(
            p
            * torch.log(
                p.clamp_min(
                    1e-8
                )
            )
        ).sum(
            dim=1
        )
        / math.log(
            TOP_K
        )
    )


def feature_cosine(
    a,
    b,
):
    return (
        (
            a
            * b
        ).sum(
            dim=1
        )
        / (
            torch.sqrt(
                (
                    a
                    * a
                ).sum(
                    dim=1
                )
                + 1e-8
            )
            * torch.sqrt(
                (
                    b
                    * b
                ).sum(
                    dim=1
                )
                + 1e-8
            )
        )
    )


def gate_features(
    r,
    retrieval,
    direct,
):
    s = r[
        "score"
    ]

    sim = r[
        "sim"
    ]

    pattern = r[
        "pattern"
    ]

    sorted_s = torch.sort(
        s,
        dim=1,
        descending=True,
    ).values

    cand_std = r[
        "cand"
    ].std(
        dim=1,
        unbiased=False,
    )

    disp_rms = torch.sqrt(
        (
            cand_std
            * cand_std
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disp_mean = cand_std.mean(
        dim=1
    )

    direct_rms = torch.sqrt(
        (
            direct
            * direct
        ).mean(
            dim=1
        )
        + 1e-8
    )

    retrieval_rms = torch.sqrt(
        (
            retrieval
            * retrieval
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disagreement = (
        retrieval
        - direct
    )

    disagreement_rms = torch.sqrt(
        (
            disagreement
            * disagreement
        ).mean(
            dim=1
        )
        + 1e-8
    )

    relative_disagreement = (
        disagreement_rms
        / (
            direct_rms
            + retrieval_rms
            + 1e-6
        )
    )

    scalars = torch.stack(
        [
            s.mean(
                dim=1
            ),
            s.std(
                dim=1,
                unbiased=False,
            ),
            s.max(
                dim=1
            ).values,
            sorted_s[
                :,
                0
            ]
            - sorted_s[
                :,
                1
            ],
            s.max(
                dim=1
            ).values
            - s.mean(
                dim=1
            ),
            score_entropy(
                s
            ),
            sim.mean(
                dim=1
            ),
            sim.std(
                dim=1,
                unbiased=False,
            ),
            sim.max(
                dim=1
            ).values,
            pattern.mean(
                dim=1
            ),
            pattern.std(
                dim=1,
                unbiased=False,
            ),
            pattern.max(
                dim=1
            ).values,
            disp_rms,
            disp_mean,
            direct_rms,
            retrieval_rms,
            disagreement_rms,
            relative_disagreement,
            feature_cosine(
                direct,
                retrieval,
            ),
        ],
        dim=1,
    )

    out = torch.cat(
        [
            r[
                "ctx"
            ],
            scalars,
        ],
        dim=1,
    )

    if out.shape[
        1
    ] != GATE_DIM:
        raise RuntimeError(
            f"Gate feature dimension mismatch: "
            f"{out.shape}"
        )

    return out


def abc_terms(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    return torch.stack(
        [
            (
                e
                * e
            ).mean(
                dim=1
            ),
            (
                e
                * delta
            ).mean(
                dim=1
            ),
            (
                delta
                * delta
            ).mean(
                dim=1
            ),
        ],
        dim=1,
    )


## 14. Fold-specific direct training

In [16]:

def fold_direct_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "fold_direct"
        ]
        / (
            f"{name}_{CURRENT_BACKBONE}_"
            f"H{horizon}_F{fold}.pt"
        )
    )


def fold_direct_history_path(
    horizon,
    fold,
):
    return (
        DIRS[
            "history"
        ]
        / (
            f"Exchange_{CURRENT_BACKBONE}_"
            f"H{horizon}_F{fold}_direct.csv"
        )
    )


def train_fold_direct(
    backbone,
    horizon,
    z,
    prefix,
    fixed_epochs,
    fold,
):
    path = fold_direct_path(
        "Exchange",
        horizon,
        fold,
    )

    r = DIRECT_RECIPES[
        backbone
    ]

    model, cfg = build_direct_model(
        backbone,
        horizon,
    )

    if (
        RESUME
        and path.is_file()
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded fold direct:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    seed = (
        r[
            "seed"
        ]
        + 10_000
        + horizon
        * 10
        + fold
    )

    set_seed(
        seed
    )

    anchors = train_anchors_for_direct(
        backbone,
        prefix,
        horizon,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=
            r[
                "learning_rate"
            ],
        weight_decay=
            r[
                "weight_decay"
            ],
    )

    scheduler = None

    if r[
        "scheduler"
    ] in {
        "TST",
        "OneCycle",
    }:
        steps_per_epoch = max(
            1,
            int(
                np.ceil(
                    len(
                        anchors
                    )
                    / r[
                        "batch_size"
                    ]
                )
            ),
        )

        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            steps_per_epoch=
                steps_per_epoch,
            pct_start=
                r[
                    "pct_start"
                ],
            # Preserve the full-model schedule budget.
            epochs=
                r[
                    "train_epochs"
                ],
            max_lr=
                r[
                    "learning_rate"
                ],
        )

    rng = np.random.default_rng(
        seed
        + 1
    )

    history = []

    for epoch in range(
        1,
        int(
            fixed_epochs
        )
        + 1,
    ):
        model.train()

        if r[
            "scheduler"
        ] == "type1":
            current_lr = set_epoch_lr_type1(
                optimizer,
                r[
                    "learning_rate"
                ],
                epoch,
            )

        order = rng.permutation(
            anchors
        )

        losses = []
        t0 = time.time()

        for left in range(
            0,
            len(
                order
            ),
            r[
                "batch_size"
            ],
        ):
            a = order[
                left:
                left
                + r[
                    "batch_size"
                ]
            ]

            if len(
                a
            ) == 0:
                continue

            x, y = make_direct_batch(
                z,
                a,
                backbone,
                horizon,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            pred, true, _ = forward_direct(
                backbone,
                model,
                x,
                y,
                horizon,
            )

            loss = F.mse_loss(
                pred,
                true,
            )

            loss.backward()
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            losses.append(
                float(
                    loss.item()
                )
            )

        if scheduler is not None:
            current_lr = optimizer.param_groups[
                0
            ][
                "lr"
            ]

        history.append({
            "Epoch":
                epoch,
            "TrainMSE":
                float(
                    np.mean(
                        losses
                    )
                ),
            "LR":
                current_lr,
            "Seconds":
                time.time()
                - t0,
        })

        pd.DataFrame(
            history
        ).to_csv(
            fold_direct_history_path(
                horizon,
                fold,
            ),
            index=False,
        )

        print(
            f"Fold {backbone:12s} "
            f"H={horizon:3d} F{fold} "
            f"ep={epoch:02d}/{fixed_epochs} | "
            f"train={history[-1]['TrainMSE']:.6f}"
        )

    model.eval()

    state = {
        k:
            v.detach()
            .cpu()
            .clone()
        for k, v
        in model.state_dict().items()
    }

    ckpt = {
        "Backbone":
            backbone,
        "Dataset":
            "Exchange",
        "Horizon":
            horizon,
        "Fold":
            fold,
        "Prefix":
            int(
                prefix
            ),
        "FixedEpochs":
            int(
                fixed_epochs
            ),
        "Seed":
            seed,
        "StateDict":
            state,
    }

    torch.save(
        ckpt,
        path,
    )

    return (
        model,
        ckpt,
    )


## 15. Efficient OOF gate-data collection

In [17]:

@torch.no_grad()
def collect_gate_data(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    z,
    anchors,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    direct_block = DIRECT_ANCHOR_BLOCK[
        CURRENT_BACKBONE
    ]

    features = []
    abcs = []
    anchors_out = []
    channels_out = []

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                z,
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        # [A, H, C] -> [A, C, H]
        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                z,
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            d = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            t = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            # Strong consistency check:
            # direct true residual must match retrieval true residual.
            max_true_diff = float(
                (
                    t
                    - r[
                        "true"
                    ]
                ).abs().max()
            )

            if (
                max_true_diff
                > 2e-5
            ):
                raise RuntimeError(
                    f"True residual mismatch: "
                    f"{max_true_diff}"
                )

            feat = gate_features(
                r,
                retrieval,
                d,
            )

            abc = abc_terms(
                d,
                retrieval,
                t,
            )

            features.append(
                feat.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            abcs.append(
                abc.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            anchors_out.append(
                pair_anchor
            )

            channels_out.append(
                pair_channel
            )

            del (
                r,
                retrieval,
                d,
                t,
                feat,
                abc,
            )

        del (
            direct_big,
            true_big,
        )

    return {
        "feature":
            np.concatenate(
                features,
                axis=0,
            ),
        "abc":
            np.concatenate(
                abcs,
                axis=0,
            ),
        "anchor":
            np.concatenate(
                anchors_out,
                axis=0,
            ),
        "channel":
            np.concatenate(
                channels_out,
                axis=0,
            ),
    }


## 16. OOF and validation caches

In [18]:

def oof_cache_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "oof"
        ]
        / (
            f"{name}_H{horizon}_"
            f"F{fold}_oof.npz"
        )
    )


def val_cache_path(
    name,
    horizon,
):
    return (
        DIRS[
            "validation"
        ]
        / (
            f"{name}_H{horizon}_"
            "validation.npz"
        )
    )


def build_oof_fold(
    data,
    horizon,
    fold,
    p0,
    p1,
    fixed_epochs,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    out_path = oof_cache_path(
        name,
        horizon,
        fold,
    )

    if (
        out_path.exists()
        and RESUME
        and not FORCE
    ):
        obj = np.load(
            out_path
        )

        print(
            "Loaded OOF cache:",
            out_path.name,
        )

        return {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    prefix = int(
        p0
        * data[
            "train_end"
        ]
    )

    oof_end = int(
        p1
        * data[
            "train_end"
        ]
    )

    z, prefix_scaler = prefix_normalize(
        data[
            "raw"
        ],
        prefix,
    )

    direct_model, _ = (
        train_fold_direct(
            CURRENT_BACKBONE,
            horizon,
            z,
            prefix,
            fixed_epochs,
            fold,
        )
    )

    retriever, _ = load_frozen_retriever(
        fold_retriever_ckpt_path(
            name,
            horizon,
            fold,
        )
    )

    memory = build_memory(
        z,
        C,
        prefix,
        horizon,
    )

    memory_gpu_obj = memory_gpu_cached(
        name,
        horizon,
        (
            f"F{fold}_"
            f"prefix{prefix}"
        ),
        retriever,
        memory,
        C,
    )

    anchors = eval_anchors(
        prefix,
        oof_end,
        horizon,
        stride=
            OOF_ANCHOR_STRIDE,
    )

    print(
        f"OOF {name} H={horizon} F{fold}: "
        f"prefix={prefix}, "
        f"end={oof_end}, "
        f"anchors={len(anchors)}, "
        f"pairs={len(anchors)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        horizon,
        direct_model,
        retriever,
        memory_gpu_obj,
        z,
        anchors,
    )

    np.savez_compressed(
        out_path,
        **out,
    )

    del (
        direct_model,
        retriever,
        memory,
        memory_gpu_obj,
        prefix_scaler,
        z,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


def build_validation_cache(
    data,
    horizon,
    direct_model,
    retriever,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    path = val_cache_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        obj = np.load(
            path
        )

        print(
            "Loaded validation cache:",
            path.name,
        )

        return {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    memory = build_memory(
        data[
            "z"
        ],
        C,
        data[
            "train_end"
        ],
        horizon,
    )

    memory_gpu_obj = memory_gpu_cached(
        name,
        horizon,
        "validation_train_memory",
        retriever,
        memory,
        C,
    )

    anchors = eval_anchors(
        data[
            "train_end"
        ],
        data[
            "val_end"
        ],
        horizon,
        stride=1,
    )

    print(
        f"Validation {name} H={horizon}: "
        f"anchors={len(anchors)}, "
        f"pairs={len(anchors)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        horizon,
        direct_model,
        retriever,
        memory_gpu_obj,
        data[
            "z"
        ],
        anchors,
    )

    np.savez_compressed(
        path,
        **out,
    )

    del (
        memory,
        memory_gpu_obj,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


## 17. Cross-fitted gate and validation-only calibration

In [19]:

def fit_feature_scaler(
    x,
):
    median = np.median(
        x,
        axis=0,
    ).astype(
        np.float32
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75
        - q25
    ).astype(
        np.float32
    )

    iqr = np.where(
        iqr < 1e-5,
        1.0,
        iqr,
    ).astype(
        np.float32
    )

    return (
        median,
        iqr,
    )


def scale_features(
    x,
    median,
    iqr,
):
    return np.clip(
        (
            x
            - median
        )
        / iqr,
        -8.0,
        8.0,
    ).astype(
        np.float32
    )


def gate_loss(
    alpha,
    abc,
):
    return (
        abc[
            :,
            0
        ]
        + 2.0
        * alpha
        * abc[
            :,
            1
        ]
        + alpha
        * alpha
        * abc[
            :,
            2
        ]
    ).mean()


def gate_checkpoint_path(
    name,
    horizon,
):
    return (
        DIRS[
            "gate"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate.pt"
        )
    )


def train_gate_epoch(
    model,
    optimizer,
    x,
    abc,
    rng,
):
    model.train()

    order = rng.permutation(
        len(
            x
        )
    )

    losses = []

    for i in range(
        0,
        len(
            order
        ),
        GATE_BATCH,
    ):
        ids = order[
            i:
            i+GATE_BATCH
        ]

        xt = torch.from_numpy(
            x[
                ids
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                ids
            ]
        ).to(
            DEVICE
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        alpha = model(
            xt
        )

        loss = gate_loss(
            alpha,
            at,
        )

        loss.backward()
        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

        del (
            xt,
            at,
            alpha,
            loss,
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def evaluate_gate(
    model,
    x,
    abc,
):
    model.eval()

    total = 0.0
    n = 0
    alpha_sum = 0.0

    for i in range(
        0,
        len(
            x
        ),
        GATE_BATCH,
    ):
        xt = torch.from_numpy(
            x[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        alpha = model(
            xt
        )

        each = (
            at[
                :,
                0
            ]
            + 2.0
            * alpha
            * at[
                :,
                1
            ]
            + alpha
            * alpha
            * at[
                :,
                2
            ]
        )

        total += float(
            each.sum()
        )

        n += len(
            alpha
        )

        alpha_sum += float(
            alpha.sum()
        )

        del (
            xt,
            at,
            alpha,
            each,
        )

    return (
        total
        / n,
        alpha_sum
        / n,
    )


def train_crossfit_gate(
    name,
    horizon,
    oof_x,
    oof_abc,
    val_x,
    val_abc,
):
    path = gate_checkpoint_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = CrossFitAdaptiveGate().to(
            DEVICE
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded gate:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    median, iqr = fit_feature_scaler(
        oof_x
    )

    train_x = scale_features(
        oof_x,
        median,
        iqr,
    )

    valid_x = scale_features(
        val_x,
        median,
        iqr,
    )

    seed = (
        CROSSFIT_SEED
        + horizon
        * 3000
        + sum(
            map(
                ord,
                name,
            )
        )
    )

    set_seed(
        seed
    )

    model = CrossFitAdaptiveGate().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    best = float(
        "inf"
    )

    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(
        1,
        GATE_MAX_EPOCHS
        + 1,
    ):
        train_mse = train_gate_epoch(
            model,
            optimizer,
            train_x,
            oof_abc,
            rng,
        )

        val_mse, mean_alpha = evaluate_gate(
            model,
            valid_x,
            val_abc,
        )

        history.append({
            "Epoch":
                epoch,
            "OOFTrainMSE":
                train_mse,
            "ValMSE":
                val_mse,
            "ValMeanAlpha":
                mean_alpha,
        })

        if (
            val_mse
            < best
            - 1e-10
        ):
            best = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        print(
            f"Gate {name:11s} H={horizon:3d} "
            f"ep={epoch:02d} "
            f"OOF={train_mse:.6f} "
            f"val={val_mse:.6f} "
            f"alpha={mean_alpha:.3f} "
            f"best={best:.6f}@{best_epoch}"
        )

        if (
            wait
            >= GATE_PATIENCE
        ):
            break

    # Reinitialize and fit only on OOF for the
    # validation-selected number of epochs.
    set_seed(
        seed
    )

    final = CrossFitAdaptiveGate().to(
        DEVICE
    )

    final_opt = torch.optim.AdamW(
        final.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    final_rng = np.random.default_rng(
        seed
        + 2
    )

    for _ in range(
        best_epoch
    ):
        train_gate_epoch(
            final,
            final_opt,
            train_x,
            oof_abc,
            final_rng,
        )

    final.eval()

    ckpt = {
        "BestEpoch":
            best_epoch,
        "BestValMSE":
            best,
        "FeatureMedian":
            median,
        "FeatureIQR":
            iqr,
        "StateDict": {
            k:
                v.detach()
                .cpu()
                .clone()
            for k, v
            in final.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        path,
    )

    pd.DataFrame(
        history
    ).to_csv(
        DIRS[
            "history"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate_history.csv"
        ),
        index=False,
    )

    return (
        final,
        ckpt,
    )


def mse_scalar(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = float(
        alpha
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_scalar(
    abc,
):
    rows = []

    best_alpha = None
    best_mse = float(
        "inf"
    )

    for alpha in ALPHA_GRID:
        mse = mse_scalar(
            abc,
            alpha,
        )

        rows.append({
            "Alpha":
                float(
                    alpha
                ),
            "MSE":
                mse,
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_alpha = float(
                alpha
            )

    return (
        best_alpha,
        pd.DataFrame(
            rows
        ),
    )


@torch.no_grad()
def gate_alpha(
    model,
    ckpt,
    x,
):
    sx = scale_features(
        x,
        ckpt[
            "FeatureMedian"
        ],
        ckpt[
            "FeatureIQR"
        ],
    )

    outputs = []

    for i in range(
        0,
        len(
            sx
        ),
        GATE_BATCH,
    ):
        t = torch.from_numpy(
            sx[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        outputs.append(
            model(
                t
            ).cpu()
            .numpy()
        )

        del t

    return np.concatenate(
        outputs
    ).astype(
        np.float32
    )


def mse_pair(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = np.asarray(
        alpha,
        dtype=np.float64,
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_lambda(
    abc,
    gate_alpha_values,
    scalar_alpha,
):
    rows = []

    best_lambda = None
    best_mse = float(
        "inf"
    )

    for lmb in LAMBDA_GRID:
        alpha = (
            (
                1.0
                - float(
                    lmb
                )
            )
            * scalar_alpha
            + float(
                lmb
            )
            * gate_alpha_values
        )

        mse = mse_pair(
            abc,
            alpha,
        )

        rows.append({
            "Lambda":
                float(
                    lmb
                ),
            "MSE":
                mse,
            "MeanAlpha":
                float(
                    alpha.mean()
                ),
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_lambda = float(
                lmb
            )

    return (
        best_lambda,
        pd.DataFrame(
            rows
        ),
    )


## 18. Final test and Oracle diagnostic

In [20]:

def empty_stat():
    return {
        "sse":
            0.0,
        "sae":
            0.0,
        "n":
            0,
    }


def update_stat(
    stat,
    pred,
    true,
):
    e = (
        pred
        - true
    )

    stat[
        "sse"
    ] += float(
        (
            e
            * e
        ).sum()
    )

    stat[
        "sae"
    ] += float(
        e.abs().sum()
    )

    stat[
        "n"
    ] += e.numel()


def finish_stat(
    stat,
):
    return (
        stat[
            "sse"
        ]
        / stat[
            "n"
        ],
        stat[
            "sae"
        ]
        / stat[
            "n"
        ],
    )


def oracle_alpha_and_prediction(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    alpha = torch.clamp(
        -(
            e
            * delta
        ).sum(
            dim=1
        )
        / (
            (
                delta
                * delta
            ).sum(
                dim=1
            )
            + 1e-8
        ),
        0.0,
        1.0,
    )

    pred = (
        direct
        + alpha[
            :,
            None
        ]
        * delta
    )

    return (
        alpha,
        pred,
    )


@torch.no_grad()
def test_evaluate(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    gate,
    gate_ckpt,
    scalar_alpha,
    shrink_lambda,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    direct_block = DIRECT_ANCHOR_BLOCK[
        CURRENT_BACKBONE
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    anchors = eval_anchors(
        data[
            "val_end"
        ],
        data[
            "test_end"
        ],
        horizon,
        stride=1,
    )

    keys = [
        "Direct",
        "Retrieval",
        "Scalar",
        "RawAdaptive",
        "ShrinkAdaptive",
        "Oracle",
    ]

    stats = {
        key:
            empty_stat()
        for key in keys
    }

    anchor_mse = {
        key:
            []
        for key in keys
    }

    channel_sse_direct = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_sse_shrink = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_count = np.zeros(
        C,
        dtype=np.int64,
    )

    raw_alpha_sum = 0.0
    shrink_alpha_sum = 0.0
    oracle_alpha_sum = 0.0
    oracle_positive = 0
    n_pairs = 0

    processed = 0

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                data[
                    "z"
                ],
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        # anchor-level MSE accumulators within this direct block
        block_anchor_sums = {
            key:
                torch.zeros(
                    len(
                        a_big
                    ),
                    device=DEVICE,
                    dtype=torch.float64,
                )
            for key in keys
        }

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                data[
                    "z"
                ],
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            direct = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            true = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            scalar = (
                direct
                + scalar_alpha
                * (
                    retrieval
                    - direct
                )
            )

            features = gate_features(
                r,
                retrieval,
                direct,
            ).cpu().numpy().astype(
                np.float32
            )

            scaled = scale_features(
                features,
                gate_ckpt[
                    "FeatureMedian"
                ],
                gate_ckpt[
                    "FeatureIQR"
                ],
            )

            gate_alpha_values = gate(
                torch.from_numpy(
                    scaled
                ).to(
                    DEVICE
                )
            )

            shrink_alpha_values = (
                (
                    1.0
                    - shrink_lambda
                )
                * scalar_alpha
                + shrink_lambda
                * gate_alpha_values
            )

            raw_adaptive = (
                direct
                + gate_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            shrink_adaptive = (
                direct
                + shrink_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            (
                oracle_alpha,
                oracle,
            ) = oracle_alpha_and_prediction(
                direct,
                retrieval,
                true,
            )

            predictions = {
                "Direct":
                    direct,
                "Retrieval":
                    retrieval,
                "Scalar":
                    scalar,
                "RawAdaptive":
                    raw_adaptive,
                "ShrinkAdaptive":
                    shrink_adaptive,
                "Oracle":
                    oracle,
            }

            for key, pred in predictions.items():
                update_stat(
                    stats[
                        key
                    ],
                    pred,
                    true,
                )

                per_anchor = (
                    (
                        (
                            pred
                            - true
                        )
                        ** 2
                    )
                    .reshape(
                        A,
                        C,
                        horizon,
                    )
                    .mean(
                        dim=(
                            1,
                            2,
                        )
                    )
                    .double()
                )

                block_anchor_sums[
                    key
                ][
                    inner:
                    inner+A
                ] = per_anchor

            direct_e2 = (
                (
                    direct
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            shrink_e2 = (
                (
                    shrink_adaptive
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            channel_sse_direct += (
                direct_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_sse_shrink += (
                shrink_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_count += (
                A
                * horizon
            )

            raw_alpha_sum += float(
                gate_alpha_values.sum()
            )

            shrink_alpha_sum += float(
                shrink_alpha_values.sum()
            )

            oracle_alpha_sum += float(
                oracle_alpha.sum()
            )

            oracle_positive += int(
                (
                    oracle_alpha
                    > 0.01
                ).sum()
            )

            n_pairs += len(
                gate_alpha_values
            )

            del (
                r,
                retrieval,
                direct,
                true,
                scalar,
                features,
                scaled,
                gate_alpha_values,
                shrink_alpha_values,
                raw_adaptive,
                shrink_adaptive,
                oracle_alpha,
                oracle,
                predictions,
                direct_e2,
                shrink_e2,
            )

        for key in keys:
            anchor_mse[
                key
            ].extend(
                block_anchor_sums[
                    key
                ].cpu()
                .numpy()
                .astype(
                    np.float32
                )
                .tolist()
            )

        processed += len(
            a_big
        )

        if (
            processed
            == len(
                a_big
            )
            or processed
            % 500
            < len(
                a_big
            )
            or processed
            == len(
                anchors
            )
        ):
            print(
                f"  test anchors "
                f"{processed}/{len(anchors)}"
            )

        del (
            direct_big,
            true_big,
            block_anchor_sums,
        )

    return {
        "anchors":
            anchors,
        "metrics": {
            key:
                finish_stat(
                    value
                )
            for key, value
            in stats.items()
        },
        "anchor_mse": {
            key:
                np.asarray(
                    value,
                    dtype=np.float32,
                )
            for key, value
            in anchor_mse.items()
        },
        "channel_direct_mse":
            channel_sse_direct
            / channel_count,
        "channel_shrink_mse":
            channel_sse_shrink
            / channel_count,
        "raw_mean_alpha":
            raw_alpha_sum
            / n_pairs,
        "shrink_mean_alpha":
            shrink_alpha_sum
            / n_pairs,
        "oracle_mean_alpha":
            oracle_alpha_sum
            / n_pairs,
        "oracle_positive_fraction":
            oracle_positive
            / n_pairs,
    }


## 19. Paired moving-block bootstrap

In [21]:

def moving_block_bootstrap(
    difference,
    n_boot=5000,
    block=24,
    seed=222222,
):
    x = np.asarray(
        difference,
        dtype=np.float64,
    )

    n = len(
        x
    )

    L = min(
        block,
        n,
    )

    rng = np.random.default_rng(
        seed
    )

    n_blocks = int(
        np.ceil(
            n
            / L
        )
    )

    max_start = max(
        1,
        n
        - L
        + 1,
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):
        starts = rng.integers(
            0,
            max_start,
            size=n_blocks,
        )

        sample = np.concatenate(
            [
                x[
                    s:
                    s+L
                ]
                for s in starts
            ]
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "MeanImprovement":
            float(
                x.mean()
            ),
        "CI_Low":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),
        "CI_High":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),
    }


# 20. Main Experiment — Exchange: 3 backbones × 4 horizons

각 backbone × horizon 조건에서:

1. full direct checkpoint load/train
2. auto-generated-or-existing frozen Exchange full retriever load
3. three chronological OOF folds
4. backbone-specific cross-fitted gate
5. validation-only scalar \(\alpha_0\)
6. validation-only shrinkage \(\lambda\)
7. test memory = train + validation
8. all stride-1 test origins
9. moving-block bootstrap
10. channel diagnostics
11. 즉시 summary 저장

을 수행합니다.

`RESUME=True`이면 기존 checkpoint와 cache를 재사용합니다.


In [22]:

existing = (
    pd.read_csv(
        SUMMARY_PATH
    )
    if (
        RESUME
        and SUMMARY_PATH.is_file()
    )
    else pd.DataFrame()
)

summary_rows = (
    existing.to_dict(
        "records"
    )
    if len(
        existing
    )
    else []
)

bootstrap_existing = (
    pd.read_csv(
        BOOTSTRAP_PATH
    )
    if (
        RESUME
        and BOOTSTRAP_PATH.is_file()
    )
    else pd.DataFrame()
)

bootstrap_rows = (
    bootstrap_existing.to_dict(
        "records"
    )
    if len(
        bootstrap_existing
    )
    else []
)


def completed(
    backbone,
    horizon,
):
    if not len(
        existing
    ):
        return False

    return bool(
        (
            (
                existing[
                    "Backbone"
                ]
                == backbone
            )
            & (
                existing[
                    "Horizon"
                ]
                == horizon
            )
        ).any()
    )


for backbone in BACKBONES:
    CURRENT_BACKBONE = (
        backbone
    )

    DIRS = backbone_dirs(
        backbone
    )

    activate_backbone(
        backbone
    )

    for horizon in HORIZONS:
        if completed(
            backbone,
            horizon,
        ):
            print(
                f"SKIP completed | "
                f"{backbone} H={horizon}"
            )
            continue

        print(
            "\n"
            + "="
            * 150
        )

        print(
            f"EXPERIMENT 28 | EXCHANGE | "
            f"{backbone} | H={horizon}"
        )

        print(
            "="
            * 150
        )

        t_condition = time.time()

        direct_model, direct_ckpt = (
            train_or_load_full_direct(
                backbone,
                horizon,
            )
        )

        retriever, retriever_ckpt = (
            load_frozen_retriever(
                full_retriever_ckpt_path(
                    "Exchange",
                    horizon,
                )
            )
        )

        # ---------------------------------------------------------
        # Chronological OOF.
        # ---------------------------------------------------------
        oof_parts = []

        for fold, (
            p0,
            p1,
        ) in enumerate(
            FOLDS,
            start=1,
        ):
            out = build_oof_fold(
                DATA[
                    "Exchange"
                ],
                horizon,
                fold,
                p0,
                p1,
                int(
                    direct_ckpt[
                        "BestEpoch"
                    ]
                ),
            )

            oof_parts.append(
                out
            )

        oof_x = np.concatenate(
            [
                p[
                    "feature"
                ]
                for p in oof_parts
            ],
            axis=0,
        ).astype(
            np.float32
        )

        oof_abc = np.concatenate(
            [
                p[
                    "abc"
                ]
                for p in oof_parts
            ],
            axis=0,
        ).astype(
            np.float32
        )

        # ---------------------------------------------------------
        # Validation-only integration selection.
        # ---------------------------------------------------------
        val = build_validation_cache(
            DATA[
                "Exchange"
            ],
            horizon,
            direct_model,
            retriever,
        )

        gate, gate_ckpt = (
            train_crossfit_gate(
                "Exchange",
                horizon,
                oof_x,
                oof_abc,
                val[
                    "feature"
                ],
                val[
                    "abc"
                ],
            )
        )

        scalar_alpha, scalar_curve = (
            choose_scalar(
                val[
                    "abc"
                ]
            )
        )

        val_gate_alpha = gate_alpha(
            gate,
            gate_ckpt,
            val[
                "feature"
            ],
        )

        shrink_lambda, lambda_curve = (
            choose_lambda(
                val[
                    "abc"
                ],
                val_gate_alpha,
                scalar_alpha,
            )
        )

        scalar_val_mse = mse_scalar(
            val[
                "abc"
            ],
            scalar_alpha,
        )

        raw_val_mse = mse_pair(
            val[
                "abc"
            ],
            val_gate_alpha,
        )

        final_val_alpha = (
            (
                1.0
                - shrink_lambda
            )
            * scalar_alpha
            + shrink_lambda
            * val_gate_alpha
        )

        shrink_val_mse = mse_pair(
            val[
                "abc"
            ],
            final_val_alpha,
        )

        calibration = (
            lambda_curve.copy()
        )

        calibration[
            "Backbone"
        ] = backbone

        calibration[
            "Dataset"
        ] = "Exchange"

        calibration[
            "Horizon"
        ] = horizon

        calibration[
            "ScalarAlpha"
        ] = scalar_alpha

        calibration.to_csv(
            DIRS[
                "calibration"
            ]
            / (
                f"Exchange_H{horizon}_"
                "lambda_curve.csv"
            ),
            index=False,
        )

        print(
            f"Validation | alpha0={scalar_alpha:.1f} | "
            f"lambda={shrink_lambda:.2f} | "
            f"scalar={scalar_val_mse:.6f} | "
            f"rawGate={raw_val_mse:.6f} | "
            f"shrink={shrink_val_mse:.6f}"
        )

        # ---------------------------------------------------------
        # Test memory: train + validation only.
        # ---------------------------------------------------------
        test_memory = build_memory(
            z_full,
            n_channels,
            val_end,
            horizon,
        )

        test_memory_gpu = memory_gpu_cached(
            "Exchange",
            horizon,
            "test_trainval_memory",
            retriever,
            test_memory,
            n_channels,
        )

        test = test_evaluate(
            DATA[
                "Exchange"
            ],
            horizon,
            direct_model,
            retriever,
            test_memory_gpu,
            gate,
            gate_ckpt,
            scalar_alpha,
            shrink_lambda,
        )

        metrics = test[
            "metrics"
        ]

        direct_mse, direct_mae = metrics[
            "Direct"
        ]

        retrieval_mse, retrieval_mae = metrics[
            "Retrieval"
        ]

        scalar_mse, scalar_mae = metrics[
            "Scalar"
        ]

        raw_mse, raw_mae = metrics[
            "RawAdaptive"
        ]

        ours_mse, ours_mae = metrics[
            "ShrinkAdaptive"
        ]

        oracle_mse, oracle_mae = metrics[
            "Oracle"
        ]

        # ---------------------------------------------------------
        # Moving-block bootstrap.
        # Positive = Direct MSE > Ours MSE.
        # ---------------------------------------------------------
        diff = (
            test[
                "anchor_mse"
            ][
                "Direct"
            ]
            - test[
                "anchor_mse"
            ][
                "ShrinkAdaptive"
            ]
        )

        boot = moving_block_bootstrap(
            diff,
            n_boot=
                BOOTSTRAP_REPLICATES,
            block=
                BOOTSTRAP_BLOCK_LEN,
            seed=
                BOOTSTRAP_SEED
                + horizon
                + 10000
                * BACKBONES.index(
                    backbone
                ),
        )

        bootstrap_rows = [
            r
            for r in bootstrap_rows
            if not (
                r.get(
                    "Backbone"
                )
                == backbone
                and int(
                    r.get(
                        "Horizon",
                        -1,
                    )
                )
                == horizon
            )
        ]

        bootstrap_rows.append({
            "Backbone":
                backbone,
            "Dataset":
                "Exchange",
            "Horizon":
                horizon,
            "Comparison":
                "Direct-ShrinkAdaptive",
            "MeanImprovement":
                boot[
                    "MeanImprovement"
                ],
            "CI_Low":
                boot[
                    "CI_Low"
                ],
            "CI_High":
                boot[
                    "CI_High"
                ],
            "SignificantPositive":
                bool(
                    boot[
                        "CI_Low"
                    ]
                    > 0
                ),
            "SignificantNegative":
                bool(
                    boot[
                        "CI_High"
                    ]
                    < 0
                ),
            "BlockLen":
                BOOTSTRAP_BLOCK_LEN,
            "Replicates":
                BOOTSTRAP_REPLICATES,
        })

        # ---------------------------------------------------------
        # Channel diagnostics.
        # ---------------------------------------------------------
        channel_df = pd.DataFrame({
            "Channel":
                np.arange(
                    n_channels
                ),
            "Direct_MSE":
                test[
                    "channel_direct_mse"
                ],
            "Ours_MSE":
                test[
                    "channel_shrink_mse"
                ],
        })

        channel_df[
            "Gain_pct"
        ] = (
            100.0
            * (
                channel_df[
                    "Direct_MSE"
                ]
                - channel_df[
                    "Ours_MSE"
                ]
            )
            / channel_df[
                "Direct_MSE"
            ]
        )

        channel_df.to_csv(
            DIRS[
                "channel"
            ]
            / (
                f"Exchange_H{horizon}_"
                "channel.csv"
            ),
            index=False,
        )

        improved_channel_fraction = float(
            (
                channel_df[
                    "Ours_MSE"
                ]
                < channel_df[
                    "Direct_MSE"
                ]
            ).mean()
        )

        oracle_headroom = (
            100.0
            * (
                direct_mse
                - oracle_mse
            )
            / direct_mse
        )

        row = {
            "Backbone":
                backbone,
            "Dataset":
                "Exchange",
            "Horizon":
                horizon,
            "DirectSeqLen":
                direct_seq_len(
                    backbone
                ),
            "RetrievalSeqLen":
                RET_SEQ_LEN,

            "Direct_MSE":
                direct_mse,
            "ShrinkAdaptive_MSE":
                ours_mse,
            "Direct_MAE":
                direct_mae,
            "ShrinkAdaptive_MAE":
                ours_mae,

            "Retrieval_MSE":
                retrieval_mse,
            "Retrieval_MAE":
                retrieval_mae,
            "Scalar_MSE":
                scalar_mse,
            "Scalar_MAE":
                scalar_mae,
            "RawAdaptive_MSE":
                raw_mse,
            "RawAdaptive_MAE":
                raw_mae,
            "Oracle_MSE":
                oracle_mse,
            "Oracle_MAE":
                oracle_mae,

            "ScalarAlpha":
                scalar_alpha,
            "ShrinkLambda":
                shrink_lambda,
            "RawMeanAlpha":
                test[
                    "raw_mean_alpha"
                ],
            "ShrinkMeanAlpha":
                test[
                    "shrink_mean_alpha"
                ],

            "MSEGain_pct":
                100.0
                * (
                    direct_mse
                    - ours_mse
                )
                / direct_mse,

            "MAEGain_pct":
                100.0
                * (
                    direct_mae
                    - ours_mae
                )
                / direct_mae,

            "OracleHeadroomFromDirect_pct":
                oracle_headroom,

            "ImprovedChannelFraction":
                improved_channel_fraction,

            "GateBestEpoch":
                gate_ckpt[
                    "BestEpoch"
                ],

            "DirectBestEpoch":
                direct_ckpt[
                    "BestEpoch"
                ],

            "OOFPairs":
                len(
                    oof_x
                ),

            "TestAnchors":
                len(
                    test[
                        "anchors"
                    ]
                ),

            "TestMemoryPerChannel":
                test_memory[
                    "M"
                ],

            "RuntimeMinutes":
                (
                    time.time()
                    - t_condition
                )
                / 60.0,
        }

        summary_rows = [
            r
            for r in summary_rows
            if not (
                r.get(
                    "Backbone"
                )
                == backbone
                and int(
                    r.get(
                        "Horizon",
                        -1,
                    )
                )
                == horizon
            )
        ]

        summary_rows.append(
            row
        )

        np.savez_compressed(
            DIRS[
                "paired"
            ]
            / (
                f"Exchange_H{horizon}_"
                "anchor_mse.npz"
            ),
            Anchors=
                test[
                    "anchors"
                ],
            Direct=
                test[
                    "anchor_mse"
                ][
                    "Direct"
                ],
            Retrieval=
                test[
                    "anchor_mse"
                ][
                    "Retrieval"
                ],
            Scalar=
                test[
                    "anchor_mse"
                ][
                    "Scalar"
                ],
            RawAdaptive=
                test[
                    "anchor_mse"
                ][
                    "RawAdaptive"
                ],
            ShrinkAdaptive=
                test[
                    "anchor_mse"
                ][
                    "ShrinkAdaptive"
                ],
            Oracle=
                test[
                    "anchor_mse"
                ][
                    "Oracle"
                ],
        )

        pd.DataFrame(
            summary_rows
        ).sort_values(
            [
                "Backbone",
                "Horizon",
            ]
        ).to_csv(
            SUMMARY_PATH,
            index=False,
        )

        pd.DataFrame(
            bootstrap_rows
        ).sort_values(
            [
                "Backbone",
                "Horizon",
            ]
        ).to_csv(
            BOOTSTRAP_PATH,
            index=False,
        )

        print(
            "\nFINAL | "
            f"{backbone} H={horizon} | "
            f"Direct={direct_mse:.6f} | "
            f"Ours={ours_mse:.6f} | "
            f"Gain={row['MSEGain_pct']:+.3f}% | "
            f"CI=[{boot['CI_Low']:.6f}, "
            f"{boot['CI_High']:.6f}]"
        )

        del (
            direct_model,
            retriever,
            gate,
            test_memory,
            test_memory_gpu,
            test,
            val,
            oof_parts,
            oof_x,
            oof_abc,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


Activated PatchTST: /code/stock_regime_retrieval/strong_forecaster/PatchTST_official/PatchTST_supervised/models/PatchTST.py

EXPERIMENT 28 | EXCHANGE | PatchTST | H=96
Loaded full direct | PatchTST H=96 | best=0.130612@6
Fold PatchTST     H= 96 F1 ep=01/6 | train=0.384075
Fold PatchTST     H= 96 F1 ep=02/6 | train=0.262540
Fold PatchTST     H= 96 F1 ep=03/6 | train=0.217716
Fold PatchTST     H= 96 F1 ep=04/6 | train=0.190683
Fold PatchTST     H= 96 F1 ep=05/6 | train=0.170967
Fold PatchTST     H= 96 F1 ep=06/6 | train=0.157542
Building memory embedding: Exchange_H96_F1_prefix2921_emb.npy (8, 342, 64)
  channel 1/8
  channel 8/8
OOF Exchange H=96 F1: prefix=2921, end=3717, anchors=176, pairs=1408, memory/C=342
Fold PatchTST     H= 96 F2 ep=01/6 | train=0.365933
Fold PatchTST     H= 96 F2 ep=02/6 | train=0.233264
Fold PatchTST     H= 96 F2 ep=03/6 | train=0.196123
Fold PatchTST     H= 96 F2 ep=04/6 | train=0.173557
Fold PatchTST     H= 96 F2 ep=05/6 | train=0.157268
Fold PatchTST     H= 

## 21. Exact 12-condition completion audit

In [23]:

summary = pd.read_csv(
    SUMMARY_PATH
)

expected = pd.MultiIndex.from_product(
    [
        BACKBONES,
        HORIZONS,
    ],
    names=[
        "Backbone",
        "Horizon",
    ],
).to_frame(
    index=False
)

present = summary[
    [
        "Backbone",
        "Horizon",
    ]
].drop_duplicates()

missing = expected.merge(
    present,
    on=[
        "Backbone",
        "Horizon",
    ],
    how="left",
    indicator=True,
)

missing = missing[
    missing[
        "_merge"
    ]
    == "left_only"
]

if len(
    missing
):
    display(
        missing
    )

    raise RuntimeError(
        "Not all 12 Exchange conditions are complete."
    )

if len(
    summary
) != 12:
    raise RuntimeError(
        f"Expected 12 summary rows, found {len(summary)}."
    )

print(
    "PASS: exactly 12 Exchange backbone × horizon conditions."
)

display(
    summary.sort_values(
        [
            "Backbone",
            "Horizon",
        ]
    )
)


PASS: exactly 12 Exchange backbone × horizon conditions.


,Backbone,Dataset,Horizon,DirectSeqLen,RetrievalSeqLen,Direct_MSE,ShrinkAdaptive_MSE,Direct_MAE,ShrinkAdaptive_MAE,Retrieval_MSE,Retrieval_MAE,Scalar_MSE,Scalar_MAE,RawAdaptive_MSE,RawAdaptive_MAE,Oracle_MSE,Oracle_MAE,ScalarAlpha,ShrinkLambda,RawMeanAlpha,ShrinkMeanAlpha,MSEGain_pct,MAEGain_pct,OracleHeadroomFromDirect_pct,ImprovedChannelFraction,GateBestEpoch,DirectBestEpoch,OOFPairs,TestAnchors,TestMemoryPerChannel,RuntimeMinutes
0,PatchTST,Exchange,96,336,96,0.088792,0.088792,0.208818,0.208818,0.156785,0.271018,0.088792,0.208818,0.088328,0.208274,0.068505,0.175894,0.0,0.0,0.100334,0.000000,0.000000,0.000000,22.847470,0.000,1,6,4224,1422,735,0.448998
1,PatchTST,Exchange,192,336,96,0.194937,0.194937,0.316183,0.316183,0.331923,0.404510,0.194937,0.316183,0.188369,0.310782,0.138208,0.256145,0.0,0.0,0.100469,0.000000,0.000000,0.000000,29.101162,0.000,1,8,3648,1326,723,0.515787
2,PatchTST,Exchange,336,336,96,0.346291,0.346291,0.428873,0.428873,0.847905,0.680416,0.346291,0.428873,0.338948,0.426695,0.241698,0.341558,0.0,0.0,0.100455,0.000000,0.000000,0.000000,30.203574,0.000,1,8,2784,1182,705,0.525618
3,PatchTST,Exchange,720,336,96,0.858390,0.858390,0.692987,0.692987,1.507078,0.982893,0.858390,0.692987,0.831744,0.686453,0.637707,0.585830,0.0,0.0,0.100451,0.000000,0.000000,0.000000,25.708968,0.000,1,9,480,798,657,0.495342
4,TimeMixer,Exchange,96,96,96,0.092543,0.090907,0.210600,0.208730,0.156785,0.271018,0.092543,0.210600,0.090033,0.207825,0.069030,0.174783,0.0,0.5,0.112030,0.056015,1.767956,0.888026,25.407261,0.875,50,4,4224,1422,735,0.245913
5,TimeMixer,Exchange,192,96,96,0.177509,0.177509,0.299693,0.299693,0.331923,0.404510,0.177509,0.299693,0.173264,0.296336,0.128363,0.246091,0.0,0.0,0.100463,0.000000,0.000000,0.000000,27.686542,0.000,1,3,3648,1326,723,0.209416
6,TimeMixer,Exchange,336,96,96,0.329322,0.329322,0.417432,0.417432,0.847905,0.680416,0.329322,0.417432,0.324291,0.417085,0.232906,0.336188,0.0,0.0,0.100468,0.000000,0.000000,0.000000,29.277118,0.000,1,4,2784,1182,705,0.221235
7,TimeMixer,Exchange,720,96,96,1.149432,1.149432,0.800125,0.800125,1.507078,0.982893,1.149432,0.800125,1.081582,0.775546,0.749514,0.636193,0.0,0.0,0.100407,0.000000,0.000000,0.000000,34.792668,0.000,1,4,480,798,657,0.186946
8,iTransformer,Exchange,96,96,96,0.098471,0.098471,0.223430,0.223430,0.156785,0.271018,0.098471,0.223430,0.095517,0.219740,0.066744,0.173314,0.0,0.0,0.100320,0.000000,0.000000,0.000000,32.219671,0.000,1,2,4224,1422,735,0.259918
9,iTransformer,Exchange,192,96,96,0.182288,0.182288,0.305967,0.305967,0.331923,0.404510,0.182288,0.305967,0.177661,0.301845,0.130050,0.248187,0.0,0.0,0.100484,0.000000,0.000000,0.000000,28.657186,0.000,1,1,3648,1326,723,0.160575


## 22. Exchange evidence summary

In [24]:

boot_df = pd.read_csv(
    BOOTSTRAP_PATH
)

boot_key = boot_df[
    boot_df[
        "Comparison"
    ]
    == "Direct-ShrinkAdaptive"
][
    [
        "Backbone",
        "Horizon",
        "CI_Low",
        "CI_High",
        "SignificantPositive",
        "SignificantNegative",
    ]
]

evidence = summary.merge(
    boot_key,
    on=[
        "Backbone",
        "Horizon",
    ],
    how="left",
)

evidence[
    "MSE_Win"
] = (
    evidence[
        "ShrinkAdaptive_MSE"
    ]
    < evidence[
        "Direct_MSE"
    ]
)

evidence[
    "MAE_Win"
] = (
    evidence[
        "ShrinkAdaptive_MAE"
    ]
    < evidence[
        "Direct_MAE"
    ]
)

backbone_summary = (
    evidence
    .groupby(
        "Backbone",
        as_index=False,
    )
    .agg(
        Conditions=(
            "Horizon",
            "size",
        ),
        MSE_Wins=(
            "MSE_Win",
            "sum",
        ),
        SignificantWins=(
            "SignificantPositive",
            "sum",
        ),
        SignificantLosses=(
            "SignificantNegative",
            "sum",
        ),
        MeanMSEGain_pct=(
            "MSEGain_pct",
            "mean",
        ),
        MAE_Wins=(
            "MAE_Win",
            "sum",
        ),
        MeanMAEGain_pct=(
            "MAEGain_pct",
            "mean",
        ),
        MeanOracleHeadroom_pct=(
            "OracleHeadroomFromDirect_pct",
            "mean",
        ),
        MeanShrinkAlpha=(
            "ShrinkMeanAlpha",
            "mean",
        ),
    )
)

display(
    backbone_summary
)

backbone_summary.to_csv(
    ROOT
    / "exchange_backbone_summary.csv",
    index=False,
)


task_summary = (
    evidence
    .groupby(
        "Horizon",
        as_index=False,
    )
    .agg(
        BackboneWins=(
            "MSE_Win",
            "sum",
        ),
        SignificantBackboneWins=(
            "SignificantPositive",
            "sum",
        ),
        SignificantBackboneLosses=(
            "SignificantNegative",
            "sum",
        ),
        MeanMSEGain_pct=(
            "MSEGain_pct",
            "mean",
        ),
    )
)

display(
    task_summary
)

task_summary.to_csv(
    ROOT
    / "exchange_horizon_three_backbone_summary.csv",
    index=False,
)


,Backbone,Conditions,MSE_Wins,SignificantWins,SignificantLosses,MeanMSEGain_pct,MAE_Wins,MeanMAEGain_pct,MeanOracleHeadroom_pct,MeanShrinkAlpha
0,PatchTST,4,0,0,0,0.000000,0,0.000000,26.965294,0.000000
1,TimeMixer,4,1,1,0,0.441989,1,0.222006,29.290897,0.014004
2,iTransformer,4,0,0,0,0.000000,0,0.000000,29.501718,0.000000


,Horizon,BackboneWins,SignificantBackboneWins,SignificantBackboneLosses,MeanMSEGain_pct
0,96,1,1,0,0.589319
1,192,0,0,0,0.000000
2,336,0,0,0,0.000000
3,720,0,0,0,0.000000


## 23. Interpretation checkpoint

Exchange는 Solar와 다른 relevance regime을 제공하므로 Experiment 28의 핵심 질문은 다음입니다.

- Solar에서 관찰한 historical-memory gain이 Exchange에서도 backbone을 넘어 반복되는가
- query-specific signal이 약한 candidate-global regime에서도 adaptive trust가 유효한가
- long-horizon \(H\in\{192,336,720\}\)에서 historical memory가 direct forecaster에 complementary한가
- global scalar와 adaptive gate의 차이가 regime에 따라 어떻게 달라지는가

기존 retrieval 분석에서 Exchange는 candidate-global / geometric-prior 성격이 강했습니다.
따라서 Solar보다 gain이 작거나 일부 horizon에서 사라져도 과학적으로 의미가 있습니다.
반대로 세 backbone에서 일관된 gain이 나온다면 forecasting-level generality를 훨씬 강하게 지지합니다.

Experiment 28 완료 후에는 Experiment 29 — Traffic으로 진행합니다.


## 24. Saved artifacts

In [25]:

print(
    "Experiment root:",
    ROOT,
)

for p in sorted(
    ROOT.rglob(
        "*"
    )
):
    if p.is_file():
        print(
            " -",
            p.relative_to(
                ROOT
            )
        )


Experiment root: /data/dataset/strong_forecaster/exchange_three_backbone_frozen_historical_memory
 - PatchTST/calibration/Exchange_H192_lambda_curve.csv
 - PatchTST/calibration/Exchange_H336_lambda_curve.csv
 - PatchTST/calibration/Exchange_H720_lambda_curve.csv
 - PatchTST/calibration/Exchange_H96_lambda_curve.csv
 - PatchTST/channel_test/Exchange_H192_channel.csv
 - PatchTST/channel_test/Exchange_H336_channel.csv
 - PatchTST/channel_test/Exchange_H720_channel.csv
 - PatchTST/channel_test/Exchange_H96_channel.csv
 - PatchTST/fold_direct/Exchange_PatchTST_H192_F1.pt
 - PatchTST/fold_direct/Exchange_PatchTST_H192_F2.pt
 - PatchTST/fold_direct/Exchange_PatchTST_H192_F3.pt
 - PatchTST/fold_direct/Exchange_PatchTST_H336_F1.pt
 - PatchTST/fold_direct/Exchange_PatchTST_H336_F2.pt
 - PatchTST/fold_direct/Exchange_PatchTST_H336_F3.pt
 - PatchTST/fold_direct/Exchange_PatchTST_H720_F1.pt
 - PatchTST/fold_direct/Exchange_PatchTST_H720_F2.pt
 - PatchTST/fold_direct/Exchange_PatchTST_H720_F3.pt
 - 